# 08 FORCE Legacy TCN Geo Stacking WITH REUSED FINAL TEST

> Saved outputs are from an earlier local run. Portable path edits have **not** been re-executed. See the repository README for setup and scientific limitations.


In [1]:
import sys
import subprocess
import importlib.util

required_packages = {
    "pandas": "pandas",
    "numpy": "numpy",
    "scikit-learn": "sklearn",
    "joblib": "joblib",
    "pyarrow": "pyarrow",
    "scipy": "scipy",
}

for package_name, module_name in required_packages.items():
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                package_name,
            ]
        )

if importlib.util.find_spec("tensorflow") is None:
    raise ImportError(
        "TensorFlow is not installed in this environment. "
        "Install a TensorFlow version compatible with your Mac/Python, "
        "restart the kernel, and rerun this notebook."
    )

print("Dependencies are available.")

Dependencies are available.


In [2]:
from pathlib import Path
from itertools import combinations
import gc
import hashlib
import json
import math
import os
import platform
import random
import time
import warnings

import joblib
import numpy as np
import pandas as pd
import scipy
import sklearn
import tensorflow as tf

from IPython.display import display
from scipy.special import expit
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    log_loss,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
)
from sklearn.preprocessing import LabelEncoder, StandardScaler

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 400)
pd.set_option("display.max_rows", 400)
pd.set_option("display.max_colwidth", 300)

RANDOM_STATE = 142
RANDOM_STATE_SPLIT = 1729
PROBABILITY_EPSILON = 1e-8

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print("TensorFlow:", tf.__version__)
print("Visible devices:", tf.config.list_physical_devices())

TensorFlow: 2.15.0
Visible devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
RUN_MODE = "pilot"  # "pilot", "full_cv", or "final_test_reuse"

ALLOW_REUSED_FINAL_TEST = False
ALLOW_REUSED_FINAL_TEST_RERUN = False

if RUN_MODE == "pilot":
    ACTIVE_OUTER_FOLDS = [1]

elif RUN_MODE == "full_cv":
    ACTIVE_OUTER_FOLDS = [
        1,
        2,
        3,
        4,
        5,
    ]

elif RUN_MODE == "final_test_reuse":
    ACTIVE_OUTER_FOLDS = []

else:
    raise ValueError(
        "RUN_MODE must be pilot, full_cv, or final_test_reuse."
    )

# Portable local paths. Override upstream data/artifacts with environment variables.
import os

def _find_repo_root():
    start = Path(os.environ.get("STRATAWEAVE_REPO_ROOT", Path.cwd())).expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").is_file() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("StrataWeave root not found. Open the repository in Jupyter or set STRATAWEAVE_REPO_ROOT.")

REPO_ROOT = _find_repo_root()
TRAIN_PATH = Path(os.environ.get("STRATAWEAVE_TRAIN_CSV", str(REPO_ROOT / "data" / "raw" / "train.csv"))).expanduser().resolve()
REUSED_FINAL_TEST_PATH = Path(os.environ.get("STRATAWEAVE_TEST_CSV", str(REPO_ROOT / "data" / "raw" / "test.csv"))).expanduser().resolve()
PROJECT_ROOT = Path(os.environ.get("STRATAWEAVE_ARTIFACT_ROOT", str(REPO_ROOT / "artifacts"))).expanduser().resolve()


FOLD_ASSIGNMENT_PATH = (
    PROJECT_ROOT
    / "01_clean_splits_support"
    / "10_grouped_5fold_well_assignment.csv"
)

LEGACY_REFERENCE_GATE_PATH = (
    PROJECT_ROOT
    / "02A_legacy_RF_exact_reproduction"
    / "09_reproduction_gate.json"
)

ARTICLE_TEST_SUMMARY_PATH = (
    PROJECT_ROOT
    / "02A_legacy_RF_exact_reproduction"
    / "01_run_summary.csv"
)

LEGACY_OOF_ROOT = (
    PROJECT_ROOT
    / "02B_legacy_RF_grouped_well_CV"
    / "full"
    / "oof_predictions"
)

STAGE_ROOT = (
    PROJECT_ROOT
    / "08_legacy_TCN_geo_support_aware_stacking"
)

FULL_CV_OUTPUT_DIR = (
    STAGE_ROOT
    / "full_cv"
)

FULL_CV_STAGE_GATE_PATH = (
    FULL_CV_OUTPUT_DIR
    / "09_stage_gate.json"
)

FULL_CV_FOLD_METRICS_PATH = (
    FULL_CV_OUTPUT_DIR
    / "01_fold_metrics.csv"
)

FULL_CV_VARIANT_SUMMARY_PATH = (
    FULL_CV_OUTPUT_DIR
    / "02_variant_summary.csv"
)

OUTPUT_DIR = (
    STAGE_ROOT
    / RUN_MODE
)

CACHE_DIR = (
    STAGE_ROOT
    / "cache"
)

MODEL_DIR = (
    OUTPUT_DIR
    / "models"
)

REUSED_TEST_SENTINEL_PATH = (
    OUTPUT_DIR
    / "00_REUSED_FINAL_TEST_CONSUMED.json"
)

for directory in [
    OUTPUT_DIR,
    CACHE_DIR,
    MODEL_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

if not TRAIN_PATH.exists():
    raise FileNotFoundError(
        TRAIN_PATH
    )

if not FOLD_ASSIGNMENT_PATH.exists():
    raise FileNotFoundError(
        FOLD_ASSIGNMENT_PATH
    )

if not LEGACY_REFERENCE_GATE_PATH.exists():
    raise FileNotFoundError(
        LEGACY_REFERENCE_GATE_PATH
    )

if RUN_MODE == "final_test_reuse":
    if not ALLOW_REUSED_FINAL_TEST:
        raise RuntimeError(
            "Set ALLOW_REUSED_FINAL_TEST=True only after full_cv is complete."
        )

    required_final_paths = [
        REUSED_FINAL_TEST_PATH,
        FULL_CV_STAGE_GATE_PATH,
        FULL_CV_FOLD_METRICS_PATH,
        FULL_CV_VARIANT_SUMMARY_PATH,
    ]

    missing_final_paths = [
        str(path)
        for path in required_final_paths
        if not path.exists()
    ]

    if missing_final_paths:
        raise FileNotFoundError(
            "Missing final-benchmark prerequisites:\n"
            + "\n".join(
                missing_final_paths
            )
        )

    if (
        REUSED_TEST_SENTINEL_PATH.exists()
        and not ALLOW_REUSED_FINAL_TEST_RERUN
    ):
        raise RuntimeError(
            "This reused final-test benchmark was already completed. "
            "Set ALLOW_REUSED_FINAL_TEST_RERUN=True only to reproduce "
            "the exact frozen run without changing the model."
        )

print("RUN_MODE:", RUN_MODE)
print("Outer folds:", ACTIVE_OUTER_FOLDS)
print("Train:", TRAIN_PATH)
print("Outputs:", OUTPUT_DIR)
print(
    "Reused final test exists:",
    REUSED_FINAL_TEST_PATH.exists(),
)
print(
    "Reused test explicitly allowed:",
    ALLOW_REUSED_FINAL_TEST,
)

RUN_MODE: pilot
Outer folds: [1]
Train: <LOCAL_PATH_REDACTED>
Outputs: <LOCAL_PATH_REDACTED>
Reused final test exists: True
Reused test explicitly allowed: False


# 2. Fixed architecture and training contract

In [4]:
# ------------------------------------------------------------
# Fixed TCN architecture
# ------------------------------------------------------------

WINDOW_LENGTH = 65
WINDOW_HALF = WINDOW_LENGTH // 2

TCN_FILTERS = 64
TCN_DILATIONS = [1, 2, 4, 8]
TCN_KERNEL_SIZE = 3
TCN_DROPOUT = 0.20
TCN_DENSE_UNITS = 128

BATCH_SIZE = 256

if RUN_MODE == "pilot":
    STEPS_PER_EPOCH_INNER = 120
    STEPS_PER_EPOCH_OUTER = 180
    MAX_EPOCHS_INNER = 12
else:
    STEPS_PER_EPOCH_INNER = 300
    STEPS_PER_EPOCH_OUTER = 450
    MAX_EPOCHS_INNER = 24

EARLY_STOPPING_PATIENCE = 5
REDUCE_LR_PATIENCE = 2
INITIAL_LEARNING_RATE = 3e-4
MIN_LEARNING_RATE = 1e-6

VALIDATION_SAMPLE_CAP = 100_000
PREDICTION_BATCH_SIZE = 512

# Class balancing for TCN sample weights.
CLASS_WEIGHT_POWER = 0.50
CLASS_WEIGHT_MIN = 0.50
CLASS_WEIGHT_MAX = 4.00

# ------------------------------------------------------------
# Exact Legacy expert
# ------------------------------------------------------------

LEGACY_RF_PARAMETERS = {
    "n_estimators": 800,
    "max_depth": 200,
    "min_samples_split": 5,
    "min_samples_leaf": 2,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbose": 1,
}

# ------------------------------------------------------------
# Geo context is fixed from the pre-test 07G development stage.
# It is not retuned here.
# ------------------------------------------------------------

GEO_K_NEIGHBORS = 3
GEO_VERTICAL_HALF_WINDOW = 60.0
GEO_MIN_ROWS_PER_WELL_WINDOW = 5
GEO_MIN_FORMATION_SUPPORT_WELLS = 2
GEO_MIN_WELLS_FULL_TRUST = 4
GEO_ROWS_PER_WELL_FULL_TRUST = 20.0
GEO_PRIOR_SMOOTHING_EQUIVALENT_WELLS = 2.0

USE_FORMATION_HIERARCHY = True

# ------------------------------------------------------------
# Meta-model
# ------------------------------------------------------------

META_LOGISTIC_C = 0.20
META_MAX_ITER = 1000
META_MAX_ROWS_PER_WELL = 6000
META_MAX_TOTAL_ROWS = 180_000

# Outputs compared in every outer fold.
MODEL_VARIANTS = [
    "legacy_only",
    "tcn_only",
    "simple_average",
    "support_aware_stacking",
]

PRIMARY_VARIANT = (
    "support_aware_stacking"
)

print("Primary variant:", PRIMARY_VARIANT)

Primary variant: support_aware_stacking


# 3. Lithology mapping and Legacy causal feature contract

In [5]:
TARGET_COLUMN = (
    "FORCE_2020_LITHOFACIES_LITHOLOGY"
)

CONFIDENCE_COLUMN = (
    "FORCE_2020_LITHOFACIES_CONFIDENCE"
)

LITHOLOGY_CODES = sorted([
    30000,
    65030,
    65000,
    80000,
    74000,
    70000,
    70032,
    88000,
    86000,
    99000,
    90000,
    93000,
])

LITHOLOGY_NAMES = {
    30000: "Sandstone",
    65000: "Shale",
    65030: "Sandstone/Shale",
    70000: "Limestone",
    70032: "Chalk",
    74000: "Dolomite",
    80000: "Marl",
    86000: "Anhydrite",
    88000: "Halite",
    90000: "Coal",
    93000: "Basement",
    99000: "Tuff",
}

label_encoder = LabelEncoder()
label_encoder.fit(
    LITHOLOGY_CODES
)

CLASS_CODES = [
    int(value)
    for value in label_encoder.classes_
]

CLASS_INDICES = list(
    range(len(CLASS_CODES))
)

CLASS_INDEX_TO_CODE = {
    index: code
    for index, code
    in enumerate(CLASS_CODES)
}

CLASS_CODE_TO_INDEX = {
    code: index
    for index, code
    in CLASS_INDEX_TO_CODE.items()
}

CLASS_INDEX_TO_NAME = {
    index: LITHOLOGY_NAMES[code]
    for index, code
    in CLASS_INDEX_TO_CODE.items()
}

CHALK_INDEX = CLASS_CODE_TO_INDEX[70032]
SHALE_INDEX = CLASS_CODE_TO_INDEX[65000]

class_mapping = pd.DataFrame({
    "class_index": CLASS_INDICES,
    "lithology_code": CLASS_CODES,
    "lithology_name": [
        CLASS_INDEX_TO_NAME[index]
        for index in CLASS_INDICES
    ],
})

display(class_mapping)

,class_index,lithology_code,lithology_name
0,0,30000,Sandstone
1,1,65000,Shale
2,2,65030,Sandstone/Shale
3,3,70000,Limestone
4,4,70032,Chalk
5,5,74000,Dolomite
6,6,80000,Marl
7,7,86000,Anhydrite
8,8,88000,Halite
9,9,90000,Coal


In [6]:
TOP_LEGACY_FEATURES = [
    "GR_NPHI_interaction",
    "GR_DTC_interaction",
    "Y_LOC",
    "GR_RHOB_RATIO",
    "NPHI_DTC_interaction",
    "X_LOC",
    "RHOB_DTC_interaction",
    "CALI",
    "Z_LOC",
    "GR_RHOB_interaction",
    "DEPTH_NORMALIZED",
    "DEPTH_MD",
    "RDEP",
    "RHOB_log",
    "RMED",
    "RHOB",
    "X_LOC",  # تکرار عمدی؛ مدل تغییر نمی‌کند
]

EXTRA_WEAK_CLASS_FEATURES = [
    "GR_flatness",
    "DTC_grad",
    "DTC_RHOB_DIFF",
    "DTC_RHOB_RATIO2",
    "RDEP_GR_RATIO",
    "RHOB_NPHI_DIFF2",
    "Acoustic_Impedance",
    "Sand_indicator",
    "Shale_indicator",
]

LEGACY_FEATURE_COLUMNS = (
    TOP_LEGACY_FEATURES
    + EXTRA_WEAK_CLASS_FEATURES
)

assert len(
    LEGACY_FEATURE_COLUMNS
) == 26

assert len(
    set(LEGACY_FEATURE_COLUMNS)
) == 25

assert (
    LEGACY_FEATURE_COLUMNS.count(
        "X_LOC"
    )
    == 2
)

print("Feature columns:", len(LEGACY_FEATURE_COLUMNS))
print("Unique names:", len(set(LEGACY_FEATURE_COLUMNS)))
print(LEGACY_FEATURE_COLUMNS)

Feature columns: 26
Unique names: 25
['GR_NPHI_interaction', 'GR_DTC_interaction', 'Y_LOC', 'GR_RHOB_RATIO', 'NPHI_DTC_interaction', 'X_LOC', 'RHOB_DTC_interaction', 'CALI', 'Z_LOC', 'GR_RHOB_interaction', 'DEPTH_NORMALIZED', 'DEPTH_MD', 'RDEP', 'RHOB_log', 'RMED', 'RHOB', 'X_LOC', 'GR_flatness', 'DTC_grad', 'DTC_RHOB_DIFF', 'DTC_RHOB_RATIO2', 'RDEP_GR_RATIO', 'RHOB_NPHI_DIFF2', 'Acoustic_Impedance', 'Sand_indicator', 'Shale_indicator']


In [7]:
TCN_FEATURES = [
    feature
    for feature in LEGACY_FEATURE_COLUMNS
    if feature not in {
        "X_LOC",
        "Y_LOC",
    }
]

# Remove duplicate names while preserving order.
TCN_FEATURES = list(
    dict.fromkeys(
        TCN_FEATURES
    )
)

assert "X_LOC" not in TCN_FEATURES
assert "Y_LOC" not in TCN_FEATURES

REQUIRED_CAUSAL_FEATURES = {
    "GR_NPHI_interaction",
    "GR_DTC_interaction",
    "GR_RHOB_RATIO",
    "NPHI_DTC_interaction",
    "RHOB_DTC_interaction",
    "GR_RHOB_interaction",
    "RHOB_log",
    "DTC_RHOB_DIFF",
    "DTC_RHOB_RATIO2",
    "RDEP_GR_RATIO",
    "RHOB_NPHI_DIFF2",
    "Acoustic_Impedance",
    "Sand_indicator",
    "Shale_indicator",
}

assert REQUIRED_CAUSAL_FEATURES.issubset(
    set(TCN_FEATURES)
)

print("TCN feature count:", len(TCN_FEATURES))
display(
    pd.DataFrame({
        "TCN_feature": TCN_FEATURES,
    })
)

TCN feature count: 23


,TCN_feature
0,GR_NPHI_interaction
1,GR_DTC_interaction
2,GR_RHOB_RATIO
3,NPHI_DTC_interaction
4,RHOB_DTC_interaction
5,CALI
6,Z_LOC
7,GR_RHOB_interaction
8,DEPTH_NORMALIZED
9,DEPTH_MD


# 4. General utilities and metrics

In [8]:
def sha256_file(
    path,
    chunk_size=1024 * 1024,
):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        while True:
            chunk = file.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def json_safe(value):
    if isinstance(value, dict):
        return {
            str(key): json_safe(item)
            for key, item in value.items()
        }

    if isinstance(
        value,
        (
            list,
            tuple,
        ),
    ):
        return [
            json_safe(item)
            for item in value
        ]

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, Path):
        return str(value)

    if (
        isinstance(value, float)
        and not np.isfinite(value)
    ):
        return None

    return value


def detect_csv_format(
    file_path,
    num_lines=5,
):
    for encoding in [
        "utf-8",
        "latin1",
        "iso-8859-1",
        "cp1252",
    ]:
        try:
            with open(
                file_path,
                "r",
                encoding=encoding,
            ) as file:
                lines = [
                    file.readline()
                    for _ in range(
                        num_lines
                    )
                ]

            if any(
                "," in line
                for line in lines
            ):
                separator = ","

            elif any(
                ";" in line
                for line in lines
            ):
                separator = ";"

            elif any(
                "\t" in line
                for line in lines
            ):
                separator = "\t"

            else:
                separator = ","

            return {
                "sep": separator,
                "encoding": encoding,
            }

        except UnicodeDecodeError:
            continue

    return {
        "sep": ",",
        "encoding": "utf-8",
    }


def load_csv_full(file_path):
    csv_format = detect_csv_format(
        file_path
    )

    print(
        f"Loading {file_path.name} "
        f"with {csv_format}"
    )

    return pd.read_csv(
        file_path,
        sep=csv_format["sep"],
        encoding=csv_format[
            "encoding"
        ],
        low_memory=False,
    )


def normalize_probability_rows(matrix):
    matrix = np.asarray(
        matrix,
        dtype=np.float64,
    )

    matrix = np.clip(
        matrix,
        PROBABILITY_EPSILON,
        np.inf,
    )

    row_sums = matrix.sum(
        axis=1,
        keepdims=True,
    )

    return matrix / row_sums


def softmax_from_log_scores(
    log_scores,
):
    log_scores = np.asarray(
        log_scores,
        dtype=np.float64,
    )

    shifted = (
        log_scores
        - log_scores.max(
            axis=1,
            keepdims=True,
        )
    )

    return normalize_probability_rows(
        np.exp(shifted)
    )


def align_probabilities(
    raw_probabilities,
    trained_classes,
):
    aligned = np.zeros(
        (
            raw_probabilities.shape[0],
            len(CLASS_INDICES),
        ),
        dtype=np.float64,
    )

    for source_column, class_index in enumerate(
        trained_classes
    ):
        aligned[
            :,
            int(class_index),
        ] = raw_probabilities[
            :,
            source_column,
        ]

    return normalize_probability_rows(
        aligned
    )


def calculate_metrics(
    y_true,
    y_pred,
    probabilities=None,
):
    y_true = np.asarray(
        y_true,
        dtype=int,
    )

    y_pred = np.asarray(
        y_pred,
        dtype=int,
    )

    present_labels = sorted(
        np.unique(y_true).tolist()
    )

    metrics = {
        "rows": int(len(y_true)),
        "accuracy": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "macro_precision_present_classes": float(
            precision_score(
                y_true,
                y_pred,
                labels=present_labels,
                average="macro",
                zero_division=0,
            )
        ),
        "macro_recall_present_classes": float(
            recall_score(
                y_true,
                y_pred,
                labels=present_labels,
                average="macro",
                zero_division=0,
            )
        ),
        "macro_f1_present_classes": float(
            f1_score(
                y_true,
                y_pred,
                labels=present_labels,
                average="macro",
                zero_division=0,
            )
        ),
        "macro_f1_all_12_classes": float(
            f1_score(
                y_true,
                y_pred,
                labels=CLASS_INDICES,
                average="macro",
                zero_division=0,
            )
        ),
        "weighted_f1": float(
            f1_score(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0,
            )
        ),
    }

    if probabilities is not None:
        metrics["log_loss"] = float(
            log_loss(
                y_true,
                probabilities,
                labels=CLASS_INDICES,
            )
        )

    return metrics


def calculate_per_class_metrics(
    y_true,
    y_pred,
):
    precision, recall, f1, support = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=CLASS_INDICES,
            zero_division=0,
        )
    )

    return pd.DataFrame({
        "class_index": CLASS_INDICES,
        "lithology_code": CLASS_CODES,
        "lithology_name": [
            CLASS_INDEX_TO_NAME[index]
            for index in CLASS_INDICES
        ],
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "support": support,
    })


def apply_legacy_chalk_refinement(
    probabilities,
):
    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    raw_predictions = np.argmax(
        probabilities,
        axis=1,
    ).astype(int)

    probability_order = np.argsort(
        probabilities,
        axis=1,
    )[:, ::-1]

    second_class = probability_order[
        :,
        1
    ]

    refined_predictions = (
        raw_predictions.copy()
    )

    refinement_mask = (
        (
            second_class
            == CHALK_INDEX
        )
        & (
            raw_predictions
            != CHALK_INDEX
        )
    )

    refined_predictions[
        refinement_mask
    ] = CHALK_INDEX

    return (
        raw_predictions,
        refined_predictions,
        refinement_mask,
    )


def calculate_per_well_metrics(
    metadata,
    probabilities,
    raw_predictions,
    refined_predictions,
):
    rows = []

    for well_name, positions_index in (
        metadata.groupby(
            "WELL"
        ).groups.items()
    ):
        positions = np.asarray(
            list(positions_index),
            dtype=int,
        )

        y_true = metadata.loc[
            positions,
            "LITHOLOGY_ENCODED",
        ].to_numpy(
            dtype=int
        )

        present_labels = sorted(
            np.unique(y_true).tolist()
        )

        rows.append({
            "WELL": well_name,
            "row_count": int(
                len(positions)
            ),
            "class_count": int(
                len(present_labels)
            ),
            "raw_accuracy": float(
                accuracy_score(
                    y_true,
                    raw_predictions[
                        positions
                    ],
                )
            ),
            "refined_accuracy": float(
                accuracy_score(
                    y_true,
                    refined_predictions[
                        positions
                    ],
                )
            ),
            "raw_balanced_accuracy": float(
                balanced_accuracy_score(
                    y_true,
                    raw_predictions[
                        positions
                    ],
                )
            ),
            "refined_balanced_accuracy": float(
                balanced_accuracy_score(
                    y_true,
                    refined_predictions[
                        positions
                    ],
                )
            ),
            "raw_macro_f1_present_classes": float(
                f1_score(
                    y_true,
                    raw_predictions[
                        positions
                    ],
                    labels=present_labels,
                    average="macro",
                    zero_division=0,
                )
            ),
            "refined_macro_f1_present_classes": float(
                f1_score(
                    y_true,
                    refined_predictions[
                        positions
                    ],
                    labels=present_labels,
                    average="macro",
                    zero_division=0,
                )
            ),
            "log_loss": float(
                log_loss(
                    y_true,
                    probabilities[
                        positions
                    ],
                    labels=CLASS_INDICES,
                )
            ),
        })

    return pd.DataFrame(rows)

In [9]:
def normalized_entropy(
    probabilities,
):
    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    probabilities = np.clip(
        probabilities,
        PROBABILITY_EPSILON,
        1.0,
    )

    entropy = -np.sum(
        probabilities
        * np.log(
            probabilities
        ),
        axis=1,
    )

    return (
        entropy
        / np.log(
            probabilities.shape[1]
        )
    )


def probability_margin(
    probabilities,
):
    sorted_probabilities = np.sort(
        np.asarray(
            probabilities,
            dtype=np.float64,
        ),
        axis=1,
    )

    return (
        sorted_probabilities[
            :,
            -1
        ]
        - sorted_probabilities[
            :,
            -2
        ]
    )


def safe_log_probabilities(
    probabilities,
):
    return np.log(
        np.clip(
            np.asarray(
                probabilities,
                dtype=np.float64,
            ),
            PROBABILITY_EPSILON,
            1.0,
        )
    )


def well_balanced_sample_weights(
    metadata,
):
    work = metadata[
        [
            "WELL",
            "LITHOLOGY_ENCODED",
        ]
    ].copy()

    well_counts = work[
        "WELL"
    ].value_counts()

    class_counts = work[
        "LITHOLOGY_ENCODED"
    ].value_counts()

    well_weight = (
        1.0
        / work[
            "WELL"
        ].map(
            well_counts
        ).to_numpy(
            dtype=np.float64
        )
    )

    class_weight = (
        1.0
        / np.sqrt(
            work[
                "LITHOLOGY_ENCODED"
            ].map(
                class_counts
            ).to_numpy(
                dtype=np.float64
            )
        )
    )

    weights = (
        well_weight
        * class_weight
    )

    weights = (
        weights
        / np.mean(
            weights
        )
    )

    return np.clip(
        weights,
        0.1,
        20.0,
    )


def subsample_meta_rows(
    metadata,
    random_state,
):
    rng = np.random.default_rng(
        random_state
    )

    selected_parts = []

    for well_name, well_frame in (
        metadata.groupby(
            "WELL",
            sort=False,
        )
    ):
        positions = well_frame.index.to_numpy(
            dtype=int
        )

        if len(
            positions
        ) > META_MAX_ROWS_PER_WELL:
            positions = rng.choice(
                positions,
                size=META_MAX_ROWS_PER_WELL,
                replace=False,
            )

        selected_parts.append(
            positions
        )

    selected = np.concatenate(
        selected_parts
    )

    if len(
        selected
    ) > META_MAX_TOTAL_ROWS:
        selected = rng.choice(
            selected,
            size=META_MAX_TOTAL_ROWS,
            replace=False,
        )

    return np.sort(
        selected
    )

# 5. Exact Legacy preprocessing and causal feature engineering

In [10]:
def fit_and_apply_train_imputers(
    dataframe,
):
    df = dataframe.copy()

    if CONFIDENCE_COLUMN in df.columns:
        df = df.drop(
            columns=[CONFIDENCE_COLUMN]
        )

    numeric_columns = (
        df.select_dtypes(
            include=[np.number]
        )
        .columns
        .tolist()
    )

    for excluded in [
        TARGET_COLUMN,
        "LITHOLOGY_ENCODED",
        "GROUP_FOLD",
    ]:
        if excluded in numeric_columns:
            numeric_columns.remove(
                excluded
            )

    imputers = {}

    for column in numeric_columns:
        if df[column].isna().any():
            column_imputer = (
                SimpleImputer(
                    strategy="median"
                )
            )

            df[column] = (
                column_imputer
                .fit_transform(
                    df[[column]]
                )
                .ravel()
            )

            imputers[column] = (
                column_imputer
            )

    df[numeric_columns] = (
        df[numeric_columns]
        .fillna(0)
        .replace(
            [np.inf, -np.inf],
            0,
        )
    )

    return (
        df,
        imputers,
    )


def apply_train_imputers_to_validation(
    dataframe,
    imputers,
):
    df = dataframe.copy()

    if CONFIDENCE_COLUMN in df.columns:
        df = df.drop(
            columns=[CONFIDENCE_COLUMN]
        )

    numeric_columns = (
        df.select_dtypes(
            include=[np.number]
        )
        .columns
        .tolist()
    )

    for excluded in [
        TARGET_COLUMN,
        "LITHOLOGY_ENCODED",
        "GROUP_FOLD",
    ]:
        if excluded in numeric_columns:
            numeric_columns.remove(
                excluded
            )

    for column in numeric_columns:
        if (
            column in imputers
            and df[column].isna().any()
        ):
            df[column] = (
                imputers[column]
                .transform(
                    df[[column]]
                )
                .ravel()
            )

    df[numeric_columns] = (
        df[numeric_columns]
        .fillna(0)
        .replace(
            [np.inf, -np.inf],
            0,
        )
    )

    return df


def add_legacy_preprocess_features(
    dataframe,
):
    df = dataframe.copy()
    epsilon = 1e-10

    if {
        "NPHI",
        "RHOB",
    }.issubset(df.columns):
        df[
            "NPHI_RHOB_RATIO"
        ] = (
            df["NPHI"]
            / (
                df["RHOB"].abs()
                + epsilon
            )
        )

    if {
        "GR",
        "RHOB",
    }.issubset(df.columns):
        df[
            "GR_RHOB_RATIO"
        ] = (
            df["GR"]
            / (
                df["RHOB"].abs()
                + epsilon
            )
        )

    if {
        "DTC",
        "RHOB",
    }.issubset(df.columns):
        df[
            "DTC_RHOB_RATIO"
        ] = (
            df["DTC"]
            / (
                df["RHOB"].abs()
                + epsilon
            )
        )

    if "DEPTH_MD" in df.columns:
        depth_min = df[
            "DEPTH_MD"
        ].min()

        depth_max = df[
            "DEPTH_MD"
        ].max()

        depth_range = (
            depth_max
            - depth_min
        )

        if depth_range > 0:
            df[
                "DEPTH_NORMALIZED"
            ] = (
                (
                    df["DEPTH_MD"]
                    - depth_min
                )
                / depth_range
            )
        else:
            df[
                "DEPTH_NORMALIZED"
            ] = 0.0

    numeric_columns = (
        df.select_dtypes(
            include=[np.number]
        )
        .columns
    )

    df[numeric_columns] = (
        df[numeric_columns]
        .fillna(0)
        .replace(
            [np.inf, -np.inf],
            0,
        )
    )

    return df


def encode_target(
    dataframe,
):
    df = dataframe.copy()

    numeric_target = (
        pd.to_numeric(
            df[TARGET_COLUMN],
            errors="raise",
        )
        .astype(int)
    )

    unknown_codes = sorted(
        set(numeric_target)
        - set(CLASS_CODES)
    )

    if unknown_codes:
        raise ValueError(
            f"Unknown lithology codes: {unknown_codes}"
        )

    df[TARGET_COLUMN] = (
        numeric_target
    )

    df[
        "LITHOLOGY_ENCODED"
    ] = label_encoder.transform(
        numeric_target
    )

    return df


def apply_legacy_exact_resampling(
    dataframe,
):
    df = dataframe.copy()

    before_counts = (
        df[TARGET_COLUMN]
        .value_counts()
        .sort_index()
    )

    def drop_fraction(
        source_df,
        class_code,
        fraction,
        random_state,
    ):
        class_mask = (
            source_df[
                TARGET_COLUMN
            ]
            == class_code
        )

        class_count = int(
            class_mask.sum()
        )

        if class_count == 0:
            return source_df

        n_to_drop = int(
            class_count
            * fraction
        )

        indices_to_drop = (
            source_df.loc[
                class_mask
            ]
            .sample(
                n=n_to_drop,
                random_state=random_state,
            )
            .index
        )

        return source_df.drop(
            indices_to_drop
        )

    df = drop_fraction(
        df,
        class_code=65000,
        fraction=0.30,
        random_state=86,
    )

    df = drop_fraction(
        df,
        class_code=30000,
        fraction=0.20,
        random_state=86,
    )

    df = drop_fraction(
        df,
        class_code=70000,
        fraction=0.20,
        random_state=142,
    )

    df = drop_fraction(
        df,
        class_code=65030,
        fraction=0.20,
        random_state=142,
    )

    # رفتار واقعی کد مقاله قبلی حفظ می‌شود.
    basement_mask = (
        df[TARGET_COLUMN]
        == 93000
    )

    dolomite_mask = (
        df[TARGET_COLUMN]
        == 74000
    )

    dolomite_count = int(
        dolomite_mask.sum()
    )

    if (
        dolomite_count > 0
        and basement_mask.sum() > 0
    ):
        dolomite_target_count = int(
            dolomite_count
            * 1.5
        )

        n_to_add = (
            dolomite_target_count
            - dolomite_count
        )

        samples_to_add = (
            df.loc[
                basement_mask
            ]
            .sample(
                n=n_to_add,
                replace=True,
                random_state=42,
            )
        )

        df = pd.concat(
            [
                df,
                samples_to_add,
            ],
            ignore_index=True,
        )

    after_counts = (
        df[TARGET_COLUMN]
        .value_counts()
        .sort_index()
    )

    audit_rows = []

    for class_code in CLASS_CODES:
        before = int(
            before_counts.get(
                class_code,
                0,
            )
        )

        after = int(
            after_counts.get(
                class_code,
                0,
            )
        )

        audit_rows.append({
            "lithology_code": class_code,
            "lithology_name": (
                LITHOLOGY_NAMES[
                    class_code
                ]
            ),
            "before_count": before,
            "after_count": after,
            "change": after - before,
            "retained_fraction": (
                after / before
                if before > 0
                else np.nan
            ),
        })

    return (
        df,
        pd.DataFrame(
            audit_rows
        ),
    )

In [11]:
BASE_CAUSAL_FEATURES = [
    "GR",
    "NPHI",
    "RHOB",
    "PEF",
    "DTC",
    "SP",
]

def compute_legacy_causal_features(
    dataframe,
):
    df = dataframe.copy()

    available = [
        feature
        for feature in BASE_CAUSAL_FEATURES
        if feature in df.columns
    ]

    for feature_1, feature_2 in combinations(
        available,
        2,
    ):
        df[
            f"{feature_1}_{feature_2}_interaction"
        ] = (
            df[feature_1]
            * df[feature_2]
        )

    for feature in available:
        df[
            f"{feature}_squared"
        ] = (
            df[feature] ** 2
        )

        df[
            f"{feature}_log"
        ] = np.log(
            np.abs(
                df[feature]
            )
            + 1
        )

    return df


def add_legacy_weak_class_features(
    dataframe,
):
    df = dataframe.copy()
    epsilon = 1e-6

    def has_columns(
        columns,
    ):
        return all(
            column in df.columns
            for column in columns
        )

    # رفتار Legacy:
    # Train بدون WELL؛ Validation با WELL
    if has_columns(["GR"]):
        if "WELL" in df.columns:
            df["GR_grad"] = (
                df.groupby(
                    "WELL"
                )["GR"]
                .diff()
                .fillna(0)
            )

            df["GR_flatness"] = (
                df.groupby(
                    "WELL"
                )["GR"]
                .rolling(
                    window=5,
                    min_periods=1,
                )
                .std()
                .reset_index(
                    level=0,
                    drop=True,
                )
            )
        else:
            df["GR_grad"] = (
                df["GR"]
                .diff()
                .fillna(0)
            )

            df["GR_flatness"] = (
                df["GR"]
                .rolling(
                    5,
                    min_periods=1,
                )
                .std()
            )

    if has_columns(["DTC"]):
        if "WELL" in df.columns:
            df["DTC_grad"] = (
                df.groupby(
                    "WELL"
                )["DTC"]
                .diff()
                .fillna(0)
            )
        else:
            df["DTC_grad"] = (
                df["DTC"]
                .diff()
                .fillna(0)
            )

    if has_columns(["RDEP"]):
        if "WELL" in df.columns:
            df["RDEP_grad"] = (
                df.groupby(
                    "WELL"
                )["RDEP"]
                .diff()
                .fillna(0)
            )
        else:
            df["RDEP_grad"] = (
                df["RDEP"]
                .diff()
                .fillna(0)
            )

    if has_columns([
        "DTC",
        "RHOB",
    ]):
        df[
            "DTC_RHOB_DIFF"
        ] = (
            df["DTC"]
            - df["RHOB"]
        )

        df[
            "DTC_RHOB_RATIO2"
        ] = (
            df["DTC"]
            / (
                df["RHOB"]
                + epsilon
            )
        )

    if has_columns([
        "RDEP",
        "GR",
    ]):
        df[
            "RDEP_GR_RATIO"
        ] = (
            df["RDEP"]
            / (
                df["GR"]
                + epsilon
            )
        )

    if has_columns([
        "RHOB",
        "NPHI",
    ]):
        df[
            "RHOB_NPHI_DIFF2"
        ] = (
            df["RHOB"]
            - df["NPHI"]
        )

    if has_columns(["GR"]):
        gr_min = df["GR"].min()
        gr_max = df["GR"].max()

        df[
            "VSH_linear"
        ] = (
            (
                df["GR"]
                - gr_min
            )
            / (
                gr_max
                - gr_min
                + epsilon
            )
        )

        df[
            "VSH_Larionov"
        ] = (
            0.083
            * (
                2 ** (
                    3.7
                    * df[
                        "VSH_linear"
                    ]
                )
                - 1
            )
        )

        df[
            "VSH_Stieber"
        ] = (
            df["VSH_linear"]
            / (
                3
                - 2
                * df[
                    "VSH_linear"
                ]
                + epsilon
            )
        )

    if has_columns(["DTC"]):
        df["Vp"] = (
            1e6
            / (
                df["DTC"]
                + epsilon
            )
        )

    if has_columns([
        "RHOB",
        "Vp",
    ]):
        df[
            "Acoustic_Impedance"
        ] = (
            df["RHOB"]
            * df["Vp"]
        )

    if has_columns([
        "NPHI",
        "RHOB",
    ]):
        df[
            "Poisson_proxy"
        ] = (
            df["NPHI"]
            / (
                df["RHOB"]
                + epsilon
            )
        )

    if "NPHI" in df.columns:
        df["Brittleness"] = (
            1
            - df["NPHI"]
        )

    if has_columns([
        "Vp",
        "RHOB",
    ]):
        df[
            "Sand_indicator"
        ] = (
            df["Vp"]
            / (
                df["RHOB"]
                + epsilon
            )
        )

    if has_columns([
        "RHOB",
        "NPHI",
        "VSH_Larionov",
    ]):
        df[
            "Shale_indicator"
        ] = (
            df["RHOB"]
            * df["NPHI"]
            * df[
                "VSH_Larionov"
            ]
        )

    numeric_columns = (
        df.select_dtypes(
            include=[np.number]
        )
        .columns
    )

    df[numeric_columns] = (
        df[numeric_columns]
        .fillna(0)
        .replace(
            [np.inf, -np.inf],
            0,
        )
    )

    return df

# 6. TCN-consistent preprocessing

In [12]:
def add_tcn_preprocess_features(
    dataframe,
    depth_min,
    depth_max,
):
    df = dataframe.copy()
    epsilon = 1e-10

    if {
        "NPHI",
        "RHOB",
    }.issubset(
        df.columns
    ):
        df[
            "NPHI_RHOB_RATIO"
        ] = (
            df[
                "NPHI"
            ]
            / (
                df[
                    "RHOB"
                ].abs()
                + epsilon
            )
        )

    if {
        "GR",
        "RHOB",
    }.issubset(
        df.columns
    ):
        df[
            "GR_RHOB_RATIO"
        ] = (
            df[
                "GR"
            ]
            / (
                df[
                    "RHOB"
                ].abs()
                + epsilon
            )
        )

    if {
        "DTC",
        "RHOB",
    }.issubset(
        df.columns
    ):
        df[
            "DTC_RHOB_RATIO"
        ] = (
            df[
                "DTC"
            ]
            / (
                df[
                    "RHOB"
                ].abs()
                + epsilon
            )
        )

    if "DEPTH_MD" in df.columns:
        denominator = max(
            float(
                depth_max
                - depth_min
            ),
            epsilon,
        )

        df[
            "DEPTH_NORMALIZED"
        ] = (
            (
                df[
                    "DEPTH_MD"
                ]
                - float(
                    depth_min
                )
            )
            / denominator
        )

    numeric_columns = (
        df.select_dtypes(
            include=[
                np.number
            ]
        )
        .columns
    )

    df[
        numeric_columns
    ] = (
        df[
            numeric_columns
        ]
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
        .fillna(
            0
        )
    )

    return df


def fit_tcn_preprocessor(
    raw_training,
):
    raw_training = (
        raw_training.copy()
        .reset_index(
            drop=True
        )
    )

    raw_training[
        "_TCN_INPUT_ORDER"
    ] = np.arange(
        len(raw_training),
        dtype=np.int64,
    )

    raw_training = (
        raw_training.sort_values(
            [
                "WELL",
                "DEPTH_MD",
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    (
        imputed,
        imputers,
    ) = fit_and_apply_train_imputers(
        raw_training
    )

    depth_min = float(
        imputed[
            "DEPTH_MD"
        ].min()
    )

    depth_max = float(
        imputed[
            "DEPTH_MD"
        ].max()
    )

    prepared = (
        add_tcn_preprocess_features(
            imputed,
            depth_min=depth_min,
            depth_max=depth_max,
        )
    )

    prepared = encode_target(
        prepared
    )

    prepared = (
        compute_legacy_causal_features(
            prepared
        )
    )

    # WELL remains available so gradient and rolling features
    # are calculated inside each well for both Train and evaluation.
    prepared = (
        add_legacy_weak_class_features(
            prepared
        )
    )

    missing_features = [
        feature
        for feature in TCN_FEATURES
        if feature not in (
            prepared.columns
        )
    ]

    if missing_features:
        raise RuntimeError(
            "Missing TCN training features: "
            + str(
                missing_features
            )
        )

    prepared = (
        prepared.sort_values(
            "_TCN_INPUT_ORDER",
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    scaler = StandardScaler()

    feature_matrix = scaler.fit_transform(
        prepared[
            TCN_FEATURES
        ]
    ).astype(
        np.float32
    )

    return {
        "frame": prepared,
        "features": feature_matrix,
        "imputers": imputers,
        "scaler": scaler,
        "depth_min": depth_min,
        "depth_max": depth_max,
    }


def transform_tcn_preprocessor(
    raw_evaluation,
    preprocessor,
):
    raw_evaluation = (
        raw_evaluation.copy()
        .reset_index(
            drop=True
        )
    )

    raw_evaluation[
        "_TCN_INPUT_ORDER"
    ] = np.arange(
        len(raw_evaluation),
        dtype=np.int64,
    )

    raw_evaluation = (
        raw_evaluation.sort_values(
            [
                "WELL",
                "DEPTH_MD",
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    imputed = (
        apply_train_imputers_to_validation(
            raw_evaluation,
            imputers=preprocessor[
                "imputers"
            ],
        )
    )

    prepared = (
        add_tcn_preprocess_features(
            imputed,
            depth_min=preprocessor[
                "depth_min"
            ],
            depth_max=preprocessor[
                "depth_max"
            ],
        )
    )

    prepared = encode_target(
        prepared
    )

    prepared = (
        compute_legacy_causal_features(
            prepared
        )
    )

    prepared = (
        add_legacy_weak_class_features(
            prepared
        )
    )

    missing_features = [
        feature
        for feature in TCN_FEATURES
        if feature not in (
            prepared.columns
        )
    ]

    if missing_features:
        raise RuntimeError(
            "Missing TCN evaluation features: "
            + str(
                missing_features
            )
        )

    prepared = (
        prepared.sort_values(
            "_TCN_INPUT_ORDER",
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    feature_matrix = (
        preprocessor[
            "scaler"
        ]
        .transform(
            prepared[
                TCN_FEATURES
            ]
        )
        .astype(
            np.float32
        )
    )

    return {
        "frame": prepared,
        "features": feature_matrix,
    }

# 7. Window bank and balanced TCN sequences

In [13]:
def build_window_bank(
    prepared_frame,
    feature_matrix,
):
    frame = (
        prepared_frame.copy()
        .reset_index(
            drop=True
        )
    )

    feature_matrix = np.asarray(
        feature_matrix,
        dtype=np.float32,
    )

    if len(
        frame
    ) != len(
        feature_matrix
    ):
        raise ValueError(
            "Frame/features row mismatch."
        )

    well_features = []
    well_labels = []
    well_names = []

    row_to_well = np.empty(
        len(
            frame
        ),
        dtype=np.int32,
    )

    row_to_local_position = np.empty(
        len(
            frame
        ),
        dtype=np.int32,
    )

    class_well_parts = {
        class_index: []
        for class_index in (
            CLASS_INDICES
        )
    }

    class_position_parts = {
        class_index: []
        for class_index in (
            CLASS_INDICES
        )
    }

    for well_index, (
        well_name,
        well_indices,
    ) in enumerate(
        frame.groupby(
            "WELL",
            sort=False,
        ).groups.items()
    ):
        original_positions = np.asarray(
            list(
                well_indices
            ),
            dtype=np.int64,
        )

        well_depth = frame.loc[
            original_positions,
            "DEPTH_MD",
        ].to_numpy(
            dtype=np.float64
        )

        local_order = np.argsort(
            well_depth,
            kind="mergesort",
        )

        sorted_positions = (
            original_positions[
                local_order
            ]
        )

        local_features = (
            feature_matrix[
                sorted_positions
            ]
        )

        local_labels = frame.loc[
            sorted_positions,
            "LITHOLOGY_ENCODED",
        ].to_numpy(
            dtype=np.int16
        )

        well_features.append(
            local_features
        )

        well_labels.append(
            local_labels
        )

        well_names.append(
            well_name
        )

        row_to_well[
            sorted_positions
        ] = well_index

        row_to_local_position[
            sorted_positions
        ] = np.arange(
            len(
                sorted_positions
            ),
            dtype=np.int32,
        )

        for class_index in (
            CLASS_INDICES
        ):
            local_class_positions = np.flatnonzero(
                local_labels
                == class_index
            ).astype(
                np.int32
            )

            if len(
                local_class_positions
            ) == 0:
                continue

            class_well_parts[
                class_index
            ].append(
                np.full(
                    len(
                        local_class_positions
                    ),
                    well_index,
                    dtype=np.int32,
                )
            )

            class_position_parts[
                class_index
            ].append(
                local_class_positions
            )

    class_references = {}

    for class_index in (
        CLASS_INDICES
    ):
        if class_well_parts[
            class_index
        ]:
            class_references[
                class_index
            ] = (
                np.concatenate(
                    class_well_parts[
                        class_index
                    ]
                ),
                np.concatenate(
                    class_position_parts[
                        class_index
                    ]
                ),
            )

        else:
            class_references[
                class_index
            ] = (
                np.empty(
                    0,
                    dtype=np.int32,
                ),
                np.empty(
                    0,
                    dtype=np.int32,
                ),
            )

    return {
        "frame": frame,
        "well_features": (
            well_features
        ),
        "well_labels": (
            well_labels
        ),
        "well_names": (
            well_names
        ),
        "row_to_well": (
            row_to_well
        ),
        "row_to_local_position": (
            row_to_local_position
        ),
        "class_references": (
            class_references
        ),
        "n_features": int(
            feature_matrix.shape[
                1
            ]
        ),
    }


def extract_window(
    bank,
    well_index,
    center_position,
):
    well_array = bank[
        "well_features"
    ][
        int(
            well_index
        )
    ]

    positions = np.arange(
        int(
            center_position
        )
        - WINDOW_HALF,
        int(
            center_position
        )
        + WINDOW_HALF
        + 1,
    )

    positions = np.clip(
        positions,
        0,
        len(
            well_array
        )
        - 1,
    )

    return well_array[
        positions
    ]


def calculate_tcn_class_weights(
    labels,
):
    counts = (
        pd.Series(
            np.asarray(
                labels,
                dtype=int,
            )
        )
        .value_counts()
        .reindex(
            CLASS_INDICES,
            fill_value=0,
        )
        .to_numpy(
            dtype=np.float64
        )
    )

    positive_counts = counts[
        counts > 0
    ]

    median_count = float(
        np.median(
            positive_counts
        )
    )

    weights = np.ones(
        len(
            CLASS_INDICES
        ),
        dtype=np.float32,
    )

    supported = counts > 0

    weights[
        supported
    ] = (
        median_count
        / counts[
            supported
        ]
    ) ** CLASS_WEIGHT_POWER

    return np.clip(
        weights,
        CLASS_WEIGHT_MIN,
        CLASS_WEIGHT_MAX,
    ).astype(
        np.float32
    )


class BalancedWindowSequence(
    tf.keras.utils.Sequence
):
    def __init__(
        self,
        bank,
        batch_size,
        steps_per_epoch,
        class_weights,
        seed,
        **kwargs,
    ):
        super().__init__(
            **kwargs
        )

        self.bank = bank
        self.batch_size = int(
            batch_size
        )
        self.steps_per_epoch = int(
            steps_per_epoch
        )
        self.class_weights = np.asarray(
            class_weights,
            dtype=np.float32,
        )

        self.rng = np.random.default_rng(
            seed
        )

        counts = np.array([
            len(
                bank[
                    "class_references"
                ][
                    class_index
                ][
                    0
                ]
            )
            for class_index in (
                CLASS_INDICES
            )
        ], dtype=np.float64)

        supported = counts > 0

        sampling_scores = np.zeros_like(
            counts
        )

        sampling_scores[
            supported
        ] = np.sqrt(
            counts[
                supported
            ]
        )

        self.class_sampling_probability = (
            sampling_scores
            / sampling_scores.sum()
        )

    def __len__(
        self,
    ):
        return self.steps_per_epoch

    def __getitem__(
        self,
        index,
    ):
        sampled_classes = (
            self.rng.choice(
                CLASS_INDICES,
                size=self.batch_size,
                replace=True,
                p=(
                    self.class_sampling_probability
                ),
            )
        )

        x_batch = np.empty(
            (
                self.batch_size,
                WINDOW_LENGTH,
                self.bank[
                    "n_features"
                ],
            ),
            dtype=np.float32,
        )

        y_batch = sampled_classes.astype(
            np.int32
        )

        sample_weights = (
            self.class_weights[
                y_batch
            ]
        )

        for batch_index, class_index in enumerate(
            sampled_classes
        ):
            wells, positions = (
                self.bank[
                    "class_references"
                ][
                    int(
                        class_index
                    )
                ]
            )

            reference_index = (
                self.rng.integers(
                    0,
                    len(
                        wells
                    ),
                )
            )

            x_batch[
                batch_index
            ] = extract_window(
                self.bank,
                wells[
                    reference_index
                ],
                positions[
                    reference_index
                ],
            )

        return (
            x_batch,
            y_batch,
            sample_weights,
        )


class FixedWindowSequence(
    tf.keras.utils.Sequence
):
    def __init__(
        self,
        bank,
        row_positions,
        batch_size,
        include_labels=True,
        **kwargs,
    ):
        super().__init__(
            **kwargs
        )

        self.bank = bank
        self.row_positions = np.asarray(
            row_positions,
            dtype=np.int64,
        )

        self.batch_size = int(
            batch_size
        )

        self.include_labels = bool(
            include_labels
        )

    def __len__(
        self,
    ):
        return int(
            np.ceil(
                len(
                    self.row_positions
                )
                / self.batch_size
            )
        )

    def __getitem__(
        self,
        index,
    ):
        start = (
            index
            * self.batch_size
        )

        end = min(
            start
            + self.batch_size,
            len(
                self.row_positions
            ),
        )

        positions = (
            self.row_positions[
                start:end
            ]
        )

        x_batch = np.empty(
            (
                len(
                    positions
                ),
                WINDOW_LENGTH,
                self.bank[
                    "n_features"
                ],
            ),
            dtype=np.float32,
        )

        for batch_index, row_position in enumerate(
            positions
        ):
            x_batch[
                batch_index
            ] = extract_window(
                self.bank,
                self.bank[
                    "row_to_well"
                ][
                    row_position
                ],
                self.bank[
                    "row_to_local_position"
                ][
                    row_position
                ],
            )

        if not self.include_labels:
            return x_batch

        y_batch = (
            self.bank[
                "frame"
            ]
            .loc[
                positions,
                "LITHOLOGY_ENCODED",
            ]
            .to_numpy(
                dtype=np.int32
            )
        )

        return (
            x_batch,
            y_batch,
        )

# 8. Multiscale TCN model

In [14]:
def residual_tcn_block(
    x,
    filters,
    kernel_size,
    dilation_rate,
    dropout_rate,
    block_name,
):
    residual = x

    x = tf.keras.layers.Conv1D(
        filters=filters,
        kernel_size=kernel_size,
        padding="same",
        dilation_rate=dilation_rate,
        use_bias=False,
        name=(
            f"{block_name}_conv1"
        ),
    )(
        x
    )

    x = tf.keras.layers.BatchNormalization(
        name=(
            f"{block_name}_bn1"
        )
    )(
        x
    )

    x = tf.keras.layers.Activation(
        "relu",
        name=(
            f"{block_name}_relu1"
        ),
    )(
        x
    )

    x = tf.keras.layers.Dropout(
        dropout_rate,
        name=(
            f"{block_name}_dropout1"
        ),
    )(
        x
    )

    x = tf.keras.layers.Conv1D(
        filters=filters,
        kernel_size=kernel_size,
        padding="same",
        dilation_rate=dilation_rate,
        use_bias=False,
        name=(
            f"{block_name}_conv2"
        ),
    )(
        x
    )

    x = tf.keras.layers.BatchNormalization(
        name=(
            f"{block_name}_bn2"
        )
    )(
        x
    )

    if (
        residual.shape[
            -1
        ]
        != filters
    ):
        residual = tf.keras.layers.Conv1D(
            filters=filters,
            kernel_size=1,
            padding="same",
            use_bias=False,
            name=(
                f"{block_name}_residual_projection"
            ),
        )(
            residual
        )

    x = tf.keras.layers.Add(
        name=(
            f"{block_name}_add"
        )
    )([
        x,
        residual,
    ])

    x = tf.keras.layers.Activation(
        "relu",
        name=(
            f"{block_name}_output_relu"
        ),
    )(
        x
    )

    return x


def build_multiscale_tcn(
    n_features,
):
    inputs = tf.keras.Input(
        shape=(
            WINDOW_LENGTH,
            n_features,
        ),
        name="windowed_causal_features",
    )

    x = tf.keras.layers.Conv1D(
        filters=TCN_FILTERS,
        kernel_size=1,
        padding="same",
        use_bias=False,
        name="input_projection",
    )(
        inputs
    )

    for block_index, dilation_rate in enumerate(
        TCN_DILATIONS
    ):
        x = residual_tcn_block(
            x,
            filters=TCN_FILTERS,
            kernel_size=(
                TCN_KERNEL_SIZE
            ),
            dilation_rate=(
                dilation_rate
            ),
            dropout_rate=(
                TCN_DROPOUT
            ),
            block_name=(
                f"tcn_block_{block_index + 1}"
            ),
        )

    center_features = tf.keras.layers.Lambda(
        lambda tensor: tensor[
            :,
            WINDOW_HALF,
            :,
        ],
        name="center_sample_features",
    )(
        x
    )

    center_features = (
        tf.keras.layers.Dense(
            TCN_DENSE_UNITS,
            activation="relu",
            name="center_dense",
        )(
            center_features
        )
    )

    center_features = (
        tf.keras.layers.Dropout(
            TCN_DROPOUT,
            name="center_dropout",
        )(
            center_features
        )
    )

    outputs = tf.keras.layers.Dense(
        len(
            CLASS_INDICES
        ),
        activation="softmax",
        name="lithology_probability",
    )(
        center_features
    )

    model = tf.keras.Model(
        inputs=inputs,
        outputs=outputs,
        name="multiscale_windowed_TCN",
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=(
                INITIAL_LEARNING_RATE
            )
        ),
        loss=(
            "sparse_categorical_crossentropy"
        ),
        metrics=[
            (
                "sparse_categorical_accuracy"
            )
        ],
    )

    return model


def sample_validation_positions(
    frame,
    cap,
    seed,
):
    rng = np.random.default_rng(
        seed
    )

    selected = []

    number_of_wells = max(
        int(
            frame[
                "WELL"
            ].nunique()
        ),
        1,
    )

    per_well_cap = max(
        int(
            cap
            / number_of_wells
        ),
        1,
    )

    for well_name, well_frame in (
        frame.groupby(
            "WELL",
            sort=False,
        )
    ):
        positions = (
            well_frame.index.to_numpy(
                dtype=np.int64
            )
        )

        if len(
            positions
        ) > per_well_cap:
            positions = rng.choice(
                positions,
                size=per_well_cap,
                replace=False,
            )

        selected.append(
            positions
        )

    selected = np.concatenate(
        selected
    )

    if len(
        selected
    ) > cap:
        selected = rng.choice(
            selected,
            size=cap,
            replace=False,
        )

    return np.sort(
        selected
    )



def train_inner_tcn(
    raw_development_train,
    raw_gate_validation,
    outer_fold,
):
    tf.keras.backend.clear_session()

    development_preprocessor = (
        fit_tcn_preprocessor(
            raw_development_train
        )
    )

    gate_transformed = (
        transform_tcn_preprocessor(
            raw_gate_validation,
            development_preprocessor,
        )
    )

    development_bank = (
        build_window_bank(
            development_preprocessor[
                "frame"
            ],
            development_preprocessor[
                "features"
            ],
        )
    )

    gate_bank = build_window_bank(
        gate_transformed[
            "frame"
        ],
        gate_transformed[
            "features"
        ],
    )

    class_weights = (
        calculate_tcn_class_weights(
            development_preprocessor[
                "frame"
            ][
                "LITHOLOGY_ENCODED"
            ].to_numpy(
                dtype=int
            )
        )
    )

    train_sequence = (
        BalancedWindowSequence(
            development_bank,
            batch_size=BATCH_SIZE,
            steps_per_epoch=(
                STEPS_PER_EPOCH_INNER
            ),
            class_weights=(
                class_weights
            ),
            seed=(
                RANDOM_STATE
                + outer_fold
            ),
        )
    )

    model = build_multiscale_tcn(
        development_bank[
            "n_features"
        ]
    )

    history = model.fit(
        train_sequence,
        epochs=(
            MAX_EPOCHS_INNER
        ),
        verbose=1,
    )

    fixed_epoch_count = int(
        MAX_EPOCHS_INNER
    )

    all_gate_positions = np.arange(
        len(
            gate_bank[
                "frame"
            ]
        ),
        dtype=np.int64,
    )

    gate_prediction_sequence = (
        FixedWindowSequence(
            gate_bank,
            row_positions=(
                all_gate_positions
            ),
            batch_size=(
                PREDICTION_BATCH_SIZE
            ),
            include_labels=False,
        )
    )

    gate_probabilities = model.predict(
        gate_prediction_sequence,
        verbose=1,
    )

    gate_probabilities = (
        normalize_probability_rows(
            gate_probabilities
        )
    )

    history_frame = pd.DataFrame(
        history.history
    )

    history_frame[
        "epoch"
    ] = np.arange(
        1,
        len(
            history_frame
        )
        + 1,
    )

    history_frame.to_csv(
        OUTPUT_DIR
        / (
            f"outer_{outer_fold}"
            "__inner_TCN_history.csv"
        ),
        index=False,
    )

    return {
        "best_epoch": (
            fixed_epoch_count
        ),
        "gate_probabilities": (
            gate_probabilities
        ),
        "gate_frame": (
            gate_bank[
                "frame"
            ]
        ),
        "class_weights": (
            class_weights
        ),
        "gate_labels_used_for_training_or_selection": False,
    }


def train_outer_tcn_and_predict(
    raw_outer_train,
    raw_outer_validation,
    best_epoch,
    outer_fold,
):
    tf.keras.backend.clear_session()

    outer_preprocessor = (
        fit_tcn_preprocessor(
            raw_outer_train
        )
    )

    outer_validation_transformed = (
        transform_tcn_preprocessor(
            raw_outer_validation,
            outer_preprocessor,
        )
    )

    outer_train_bank = (
        build_window_bank(
            outer_preprocessor[
                "frame"
            ],
            outer_preprocessor[
                "features"
            ],
        )
    )

    outer_validation_bank = (
        build_window_bank(
            outer_validation_transformed[
                "frame"
            ],
            outer_validation_transformed[
                "features"
            ],
        )
    )

    class_weights = (
        calculate_tcn_class_weights(
            outer_preprocessor[
                "frame"
            ][
                "LITHOLOGY_ENCODED"
            ].to_numpy(
                dtype=int
            )
        )
    )

    train_sequence = (
        BalancedWindowSequence(
            outer_train_bank,
            batch_size=BATCH_SIZE,
            steps_per_epoch=(
                STEPS_PER_EPOCH_OUTER
            ),
            class_weights=(
                class_weights
            ),
            seed=(
                RANDOM_STATE
                + 100
                + outer_fold
            ),
        )
    )

    model = build_multiscale_tcn(
        outer_train_bank[
            "n_features"
        ]
    )

    model.fit(
        train_sequence,
        epochs=max(
            int(
                best_epoch
            ),
            1,
        ),
        verbose=1,
    )

    prediction_positions = np.arange(
        len(
            outer_validation_bank[
                "frame"
            ]
        ),
        dtype=np.int64,
    )

    prediction_sequence = (
        FixedWindowSequence(
            outer_validation_bank,
            row_positions=(
                prediction_positions
            ),
            batch_size=(
                PREDICTION_BATCH_SIZE
            ),
            include_labels=False,
        )
    )

    probabilities = model.predict(
        prediction_sequence,
        verbose=1,
    )

    probabilities = (
        normalize_probability_rows(
            probabilities
        )
    )

    weights_path = (
        MODEL_DIR
        / (
            f"outer_{outer_fold}"
            "__outer_TCN.weights.h5"
        )
    )

    model.save_weights(
        weights_path
    )

    return {
        "probabilities": (
            probabilities
        ),
        "frame": (
            outer_validation_bank[
                "frame"
            ]
        ),
        "weights_path": str(
            weights_path
        ),
    }

# 9. Exact Legacy full-RF expert

In [15]:
def train_exact_legacy_full_and_predict(
    raw_training,
    raw_evaluation,
):
    raw_training = (
        raw_training.copy()
        .reset_index(
            drop=True
        )
    )

    raw_evaluation = (
        raw_evaluation.copy()
        .reset_index(
            drop=True
        )
    )

    (
        train_imputed,
        train_imputers,
    ) = fit_and_apply_train_imputers(
        raw_training
    )

    train_preprocessed = (
        add_legacy_preprocess_features(
            train_imputed
        )
    )

    (
        train_resampled,
        resampling_audit,
    ) = apply_legacy_exact_resampling(
        train_preprocessed
    )

    train_resampled = encode_target(
        train_resampled
    )

    train_causal = (
        compute_legacy_causal_features(
            train_resampled
        )
    )

    exclude_columns = {
        "WELL",
        TARGET_COLUMN,
        CONFIDENCE_COLUMN,
        "LITHOLOGY_ENCODED",
        "GROUP",
        "FORMATION",
        "ORIGINAL_ROW_ID",
    }

    candidate_numeric_features = [
        column
        for column in train_causal.columns
        if (
            column not in (
                exclude_columns
            )
            and pd.api.types.is_numeric_dtype(
                train_causal[
                    column
                ]
            )
        )
    ]

    # Exact Legacy behavior:
    # WELL is removed before weak-feature engineering in Train.
    train_numeric_only = (
        train_causal[
            candidate_numeric_features
        ]
        .copy()
    )

    train_with_weak = (
        add_legacy_weak_class_features(
            train_numeric_only
        )
    )

    missing_train_features = [
        feature
        for feature in set(
            LEGACY_FEATURE_COLUMNS
        )
        if feature not in (
            train_with_weak.columns
        )
    ]

    if missing_train_features:
        raise RuntimeError(
            "Missing Legacy Train features: "
            + str(
                missing_train_features
            )
        )

    X_train = (
        train_with_weak[
            LEGACY_FEATURE_COLUMNS
        ]
        .copy()
    )

    y_train = (
        train_causal[
            "LITHOLOGY_ENCODED"
        ]
        .astype(
            int
        )
        .to_numpy()
    )

    evaluation_imputed = (
        apply_train_imputers_to_validation(
            raw_evaluation,
            imputers=(
                train_imputers
            ),
        )
    )

    evaluation_preprocessed = (
        add_legacy_preprocess_features(
            evaluation_imputed
        )
    )

    evaluation_preprocessed = (
        encode_target(
            evaluation_preprocessed
        )
    )

    evaluation_causal = (
        compute_legacy_causal_features(
            evaluation_preprocessed
        )
    )

    # Exact Legacy behavior:
    # WELL remains during weak-feature engineering in evaluation.
    evaluation_with_weak = (
        add_legacy_weak_class_features(
            evaluation_causal
        )
    )

    missing_evaluation_features = [
        feature
        for feature in set(
            LEGACY_FEATURE_COLUMNS
        )
        if feature not in (
            evaluation_with_weak.columns
        )
    ]

    if missing_evaluation_features:
        raise RuntimeError(
            "Missing Legacy evaluation features: "
            + str(
                missing_evaluation_features
            )
        )

    X_evaluation = (
        evaluation_with_weak[
            LEGACY_FEATURE_COLUMNS
        ]
        .copy()
    )

    evaluation_metadata_columns = [
        "WELL",
        "DEPTH_MD",
        TARGET_COLUMN,
        "LITHOLOGY_ENCODED",
    ]

    for optional_column in [
        "X_LOC",
        "Y_LOC",
        "Z_LOC",
        "FORMATION",
        "ORIGINAL_ROW_ID",
    ]:
        if optional_column in (
            evaluation_with_weak.columns
        ):
            evaluation_metadata_columns.append(
                optional_column
            )

    evaluation_metadata = (
        evaluation_with_weak[
            evaluation_metadata_columns
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    scaler = StandardScaler()

    X_train_scaled = (
        scaler.fit_transform(
            X_train
        )
    )

    X_evaluation_scaled = (
        scaler.transform(
            X_evaluation
        )
    )

    model = RandomForestClassifier(
        **LEGACY_RF_PARAMETERS
    )

    model.fit(
        X_train_scaled,
        y_train,
    )

    partial_probabilities = (
        model.predict_proba(
            X_evaluation_scaled
        )
    )

    probabilities = align_probabilities(
        partial_probabilities,
        model.classes_,
    )

    return {
        "probabilities": (
            probabilities
        ),
        "metadata": (
            evaluation_metadata
        ),
        "resampling_audit": (
            resampling_audit
        ),
    }


def legacy_outer_oof_path(
    outer_fold,
):
    return (
        LEGACY_OOF_ROOT
        / (
            "legacy_exact"
            f"__outer_fold_{outer_fold}"
            "__oof.parquet"
        )
    )


def load_legacy_outer_oof(
    outer_fold,
    raw_outer_validation,
):
    path = legacy_outer_oof_path(
        outer_fold
    )

    if not path.exists():
        raise FileNotFoundError(
            "Exact Legacy OOF prediction is missing. "
            "Run 02B full grouped CV first:\n"
            + str(
                path
            )
        )

    oof = pd.read_parquet(
        path
    ).reset_index(
        drop=True
    )

    raw_outer_validation = (
        raw_outer_validation.copy()
        .reset_index(
            drop=True
        )
    )

    if len(
        oof
    ) != len(
        raw_outer_validation
    ):
        raise RuntimeError(
            "Legacy OOF row count does not match outer validation."
        )

    same_well = np.array_equal(
        oof[
            "WELL"
        ].astype(
            str
        ).to_numpy(),
        raw_outer_validation[
            "WELL"
        ].astype(
            str
        ).to_numpy(),
    )

    same_depth = np.allclose(
        oof[
            "DEPTH_MD"
        ].to_numpy(
            dtype=np.float64
        ),
        raw_outer_validation[
            "DEPTH_MD"
        ].to_numpy(
            dtype=np.float64
        ),
        equal_nan=True,
    )

    same_target = np.array_equal(
        pd.to_numeric(
            oof[
                TARGET_COLUMN
            ]
        ).astype(
            int
        ).to_numpy(),
        pd.to_numeric(
            raw_outer_validation[
                TARGET_COLUMN
            ]
        ).astype(
            int
        ).to_numpy(),
    )

    if not (
        same_well
        and same_depth
        and same_target
    ):
        raise RuntimeError(
            "Legacy OOF rows are not aligned with outer validation."
        )

    probability_columns = [
        f"prob_{class_code}"
        for class_code in (
            CLASS_CODES
        )
    ]

    probabilities = (
        oof[
            probability_columns
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    return {
        "probabilities": (
            normalize_probability_rows(
                probabilities
            )
        ),
        "frame": oof,
        "path": str(
            path
        ),
    }

# 10. Geo-stratigraphic support context

In [16]:
MIN_ROWS_PER_WELL_WINDOW = (
    GEO_MIN_ROWS_PER_WELL_WINDOW
)

MIN_FORMATION_SUPPORT_WELLS = (
    GEO_MIN_FORMATION_SUPPORT_WELLS
)

MIN_WELLS_FULL_TRUST = (
    GEO_MIN_WELLS_FULL_TRUST
)

ROWS_PER_WELL_FULL_TRUST = (
    GEO_ROWS_PER_WELL_FULL_TRUST
)

PRIOR_SMOOTHING_EQUIVALENT_WELLS = (
    GEO_PRIOR_SMOOTHING_EQUIVALENT_WELLS
)

In [17]:
def clean_numeric_series(series):
    values = pd.to_numeric(
        series,
        errors="coerce",
    ).astype(float)

    values.loc[
        ~np.isfinite(
            values.to_numpy(
                dtype=float
            )
        )
    ] = np.nan

    return values


def choose_vertical_column(
    training_dataframe,
):
    for column in [
        "Z_LOC",
        "DEPTH_MD",
    ]:
        if column not in (
            training_dataframe.columns
        ):
            continue

        values = clean_numeric_series(
            training_dataframe[
                column
            ]
        )

        coverage = float(
            values.notna().mean()
        )

        unique_count = int(
            values.nunique(
                dropna=True
            )
        )

        if (
            coverage >= 0.80
            and unique_count >= 100
        ):
            return {
                "vertical_column": column,
                "coverage": coverage,
                "unique_count": unique_count,
            }

    raise RuntimeError(
        "Neither Z_LOC nor DEPTH_MD has sufficient support."
    )


def create_prefix_profile(
    z_values,
    encoded_labels,
):
    z_values = np.asarray(
        z_values,
        dtype=np.float64,
    )

    encoded_labels = np.asarray(
        encoded_labels,
        dtype=np.int16,
    )

    order = np.argsort(
        z_values,
        kind="mergesort",
    )

    z_sorted = z_values[
        order
    ]

    labels_sorted = encoded_labels[
        order
    ]

    one_hot = np.zeros(
        (
            len(z_sorted),
            len(CLASS_INDICES),
        ),
        dtype=np.int32,
    )

    one_hot[
        np.arange(
            len(z_sorted)
        ),
        labels_sorted,
    ] = 1

    prefix = np.zeros(
        (
            len(z_sorted) + 1,
            len(CLASS_INDICES),
        ),
        dtype=np.int32,
    )

    prefix[
        1:
    ] = np.cumsum(
        one_hot,
        axis=0,
        dtype=np.int32,
    )

    return {
        "z": z_sorted,
        "prefix": prefix,
        "rows": int(len(z_sorted)),
    }


def counts_from_profile(
    profile,
    query_z,
    half_window,
):
    query_z = np.asarray(
        query_z,
        dtype=np.float64,
    )

    z_sorted = profile[
        "z"
    ]

    prefix = profile[
        "prefix"
    ]

    left = np.searchsorted(
        z_sorted,
        query_z
        - half_window,
        side="left",
    )

    right = np.searchsorted(
        z_sorted,
        query_z
        + half_window,
        side="right",
    )

    return (
        prefix[
            right
        ]
        - prefix[
            left
        ]
    ).astype(
        np.float64,
        copy=False,
    )


def build_geo_prior_bank(
    raw_training,
):
    vertical_info = (
        choose_vertical_column(
            raw_training
        )
    )

    vertical_column = (
        vertical_info[
            "vertical_column"
        ]
    )

    required_columns = {
        "WELL",
        "X_LOC",
        "Y_LOC",
        vertical_column,
        TARGET_COLUMN,
    }

    missing_columns = (
        required_columns
        - set(raw_training.columns)
    )

    if missing_columns:
        raise KeyError(
            "Missing geo-prior columns: "
            + str(
                sorted(
                    missing_columns
                )
            )
        )

    work = raw_training.copy()

    work[
        "X_LOC"
    ] = clean_numeric_series(
        work[
            "X_LOC"
        ]
    )

    work[
        "Y_LOC"
    ] = clean_numeric_series(
        work[
            "Y_LOC"
        ]
    )

    work[
        vertical_column
    ] = clean_numeric_series(
        work[
            vertical_column
        ]
    )

    work[
        TARGET_COLUMN
    ] = pd.to_numeric(
        work[
            TARGET_COLUMN
        ],
        errors="raise",
    ).astype(int)

    work[
        "LITHOLOGY_ENCODED"
    ] = label_encoder.transform(
        work[
            TARGET_COLUMN
        ]
    )

    coordinate_table = (
        work.groupby(
            "WELL",
            as_index=False,
        )
        .agg(
            X_LOC=(
                "X_LOC",
                "median",
            ),
            Y_LOC=(
                "Y_LOC",
                "median",
            ),
        )
    )

    coordinate_table = (
        coordinate_table.loc[
            coordinate_table[
                [
                    "X_LOC",
                    "Y_LOC",
                ]
            ]
            .notna()
            .all(axis=1)
        ]
        .reset_index(drop=True)
    )

    profile_work = (
        work.loc[
            work[
                vertical_column
            ].notna()
        ]
        .copy()
    )

    all_profiles = {}

    for well_name, well_frame in (
        profile_work.groupby(
            "WELL",
            sort=False,
        )
    ):
        all_profiles[
            well_name
        ] = create_prefix_profile(
            well_frame[
                vertical_column
            ].to_numpy(
                dtype=float
            ),
            well_frame[
                "LITHOLOGY_ENCODED"
            ].to_numpy(
                dtype=int
            ),
        )

    formation_profiles = {}

    formation_available = bool(
        USE_FORMATION_HIERARCHY
        and "FORMATION" in (
            profile_work.columns
        )
    )

    if formation_available:
        formation_work = (
            profile_work.loc[
                profile_work[
                    "FORMATION"
                ].notna()
            ]
            .copy()
        )

        formation_work[
            "FORMATION_KEY"
        ] = (
            formation_work[
                "FORMATION"
            ]
            .astype(str)
        )

        for (
            well_name,
            formation_key,
        ), group_frame in (
            formation_work.groupby(
                [
                    "WELL",
                    "FORMATION_KEY",
                ],
                sort=False,
            )
        ):
            formation_profiles[
                (
                    well_name,
                    formation_key,
                )
            ] = create_prefix_profile(
                group_frame[
                    vertical_column
                ].to_numpy(
                    dtype=float
                ),
                group_frame[
                    "LITHOLOGY_ENCODED"
                ].to_numpy(
                    dtype=int
                ),
            )

    per_well_class_distribution = (
        pd.crosstab(
            work[
                "WELL"
            ],
            work[
                "LITHOLOGY_ENCODED"
            ],
        )
        .reindex(
            columns=CLASS_INDICES,
            fill_value=0,
        )
        .astype(float)
    )

    per_well_class_distribution = (
        per_well_class_distribution.div(
            per_well_class_distribution.sum(
                axis=1
            ),
            axis=0,
        )
    )

    global_well_balanced_prior = (
        per_well_class_distribution.mean(
            axis=0
        )
        .to_numpy(
            dtype=np.float64
        )
    )

    global_well_balanced_prior = (
        normalize_probability_rows(
            global_well_balanced_prior[
                None,
                :
            ]
        )[
            0
        ]
    )

    coordinate_matrix = (
        coordinate_table[
            [
                "X_LOC",
                "Y_LOC",
            ]
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    if len(coordinate_matrix) >= 2:
        pairwise = (
            coordinate_matrix[
                :,
                None,
                :
            ]
            - coordinate_matrix[
                None,
                :,
                :
            ]
        )

        distances = np.sqrt(
            np.sum(
                pairwise
                ** 2,
                axis=2,
            )
        )

        np.fill_diagonal(
            distances,
            np.inf,
        )

        nearest_distances = (
            distances.min(
                axis=1
            )
        )

        median_nearest_distance = float(
            np.median(
                nearest_distances[
                    np.isfinite(
                        nearest_distances
                    )
                ]
            )
        )

    else:
        median_nearest_distance = 1.0

    return {
        "vertical_column": vertical_column,
        "vertical_info": vertical_info,
        "coordinate_table": (
            coordinate_table
        ),
        "all_profiles": all_profiles,
        "formation_profiles": (
            formation_profiles
        ),
        "formation_available": (
            formation_available
        ),
        "global_well_balanced_prior": (
            global_well_balanced_prior
        ),
        "median_nearest_distance": max(
            median_nearest_distance,
            PROBABILITY_EPSILON,
        ),
        "training_wells": int(
            work[
                "WELL"
            ].nunique()
        ),
    }

In [18]:
def aggregate_neighbor_profiles(
    query_z,
    neighbor_wells,
    neighbor_weights,
    profile_lookup,
    half_window,
):
    query_z = np.asarray(
        query_z,
        dtype=np.float64,
    )

    n_rows = len(query_z)

    weighted_sum = np.zeros(
        (
            n_rows,
            len(CLASS_INDICES),
        ),
        dtype=np.float64,
    )

    total_weight = np.zeros(
        n_rows,
        dtype=np.float64,
    )

    support_wells = np.zeros(
        n_rows,
        dtype=np.int16,
    )

    weighted_row_support = np.zeros(
        n_rows,
        dtype=np.float64,
    )

    contribution_records = []

    for well_name, well_weight in zip(
        neighbor_wells,
        neighbor_weights,
    ):
        profile = profile_lookup.get(
            well_name
        )

        if profile is None:
            continue

        counts = counts_from_profile(
            profile,
            query_z,
            half_window,
        )

        totals = counts.sum(axis=1)

        supported = (
            totals
            >= MIN_ROWS_PER_WELL_WINDOW
        )

        if not supported.any():
            continue

        proportions = np.zeros_like(
            counts,
            dtype=np.float64,
        )

        proportions[
            supported
        ] = (
            counts[
                supported
            ]
            / totals[
                supported,
                None,
            ]
        )

        effective_weight = (
            float(well_weight)
            * supported.astype(
                np.float64
            )
        )

        weighted_sum += (
            proportions
            * effective_weight[
                :,
                None,
            ]
        )

        total_weight += effective_weight

        support_wells += (
            supported.astype(
                np.int16
            )
        )

        weighted_row_support += (
            totals
            * effective_weight
        )

        contribution_records.append({
            "proportions": proportions,
            "effective_weight": (
                effective_weight
            ),
        })

    aggregate = np.zeros_like(
        weighted_sum
    )

    valid = total_weight > 0

    aggregate[
        valid
    ] = (
        weighted_sum[
            valid
        ]
        / total_weight[
            valid,
            None,
        ]
    )

    average_rows = np.zeros(
        n_rows,
        dtype=np.float64,
    )

    average_rows[
        valid
    ] = (
        weighted_row_support[
            valid
        ]
        / total_weight[
            valid
        ]
    )

    dispersion = np.ones(
        n_rows,
        dtype=np.float64,
    )

    if contribution_records:
        weighted_dispersion = np.zeros(
            n_rows,
            dtype=np.float64,
        )

        for record in (
            contribution_records
        ):
            total_variation = (
                0.5
                * np.abs(
                    record[
                        "proportions"
                    ]
                    - aggregate
                ).sum(axis=1)
            )

            weighted_dispersion += (
                total_variation
                * record[
                    "effective_weight"
                ]
            )

        dispersion[
            valid
        ] = (
            weighted_dispersion[
                valid
            ]
            / total_weight[
                valid
            ]
        )

    agreement = np.clip(
        1.0 - dispersion,
        0.0,
        1.0,
    )

    return {
        "aggregate": aggregate,
        "valid": valid,
        "support_wells": (
            support_wells
        ),
        "average_rows": (
            average_rows
        ),
        "agreement": agreement,
    }


def compute_geo_prior(
    raw_query,
    bank,
    k_neighbors,
    vertical_half_window,
):
    query = (
        raw_query.copy()
        .reset_index(drop=True)
    )

    vertical_column = (
        bank[
            "vertical_column"
        ]
    )

    required_query_columns = {
        "WELL",
        "X_LOC",
        "Y_LOC",
        vertical_column,
    }

    missing_query_columns = (
        required_query_columns
        - set(query.columns)
    )

    if missing_query_columns:
        raise KeyError(
            "Missing query geo-prior columns: "
            + str(
                sorted(
                    missing_query_columns
                )
            )
        )

    query[
        "X_LOC"
    ] = clean_numeric_series(
        query[
            "X_LOC"
        ]
    )

    query[
        "Y_LOC"
    ] = clean_numeric_series(
        query[
            "Y_LOC"
        ]
    )

    query[
        vertical_column
    ] = clean_numeric_series(
        query[
            vertical_column
        ]
    )

    n_rows = len(query)

    prior_matrix = np.tile(
        bank[
            "global_well_balanced_prior"
        ][
            None,
            :
        ],
        (
            n_rows,
            1,
        ),
    ).astype(
        np.float64
    )

    reliability_base = np.zeros(
        n_rows,
        dtype=np.float64,
    )

    nearest_distance_output = np.full(
        n_rows,
        np.nan,
        dtype=np.float64,
    )

    support_wells_output = np.zeros(
        n_rows,
        dtype=np.int16,
    )

    average_rows_output = np.zeros(
        n_rows,
        dtype=np.float64,
    )

    agreement_output = np.zeros(
        n_rows,
        dtype=np.float64,
    )

    formation_specific_output = np.zeros(
        n_rows,
        dtype=bool,
    )

    coordinate_table = (
        bank[
            "coordinate_table"
        ]
    )

    training_coordinates = (
        coordinate_table[
            [
                "X_LOC",
                "Y_LOC",
            ]
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    training_well_names = (
        coordinate_table[
            "WELL"
        ]
        .to_numpy()
    )

    if len(training_well_names) == 0:
        return {
            "prior": prior_matrix,
            "reliability_base": (
                reliability_base
            ),
            "nearest_distance": (
                nearest_distance_output
            ),
            "support_wells": (
                support_wells_output
            ),
            "average_rows": (
                average_rows_output
            ),
            "agreement": (
                agreement_output
            ),
            "formation_specific": (
                formation_specific_output
            ),
        }

    for query_well, row_indices in (
        query.groupby(
            "WELL",
            sort=False,
        ).groups.items()
    ):
        positions = np.asarray(
            list(row_indices),
            dtype=int,
        )

        well_x = float(
            query.loc[
                positions,
                "X_LOC",
            ].median()
        )

        well_y = float(
            query.loc[
                positions,
                "Y_LOC",
            ].median()
        )

        if (
            not np.isfinite(well_x)
            or not np.isfinite(well_y)
        ):
            continue

        differences = (
            training_coordinates
            - np.array([
                well_x,
                well_y,
            ])[
                None,
                :
            ]
        )

        distances = np.sqrt(
            np.sum(
                differences
                ** 2,
                axis=1,
            )
        )

        distances[
            training_well_names
            == query_well
        ] = np.inf

        finite_order = np.argsort(
            distances
        )

        finite_order = finite_order[
            np.isfinite(
                distances[
                    finite_order
                ]
            )
        ]

        if len(finite_order) == 0:
            continue

        selected_order = finite_order[
            :min(
                int(k_neighbors),
                len(finite_order),
            )
        ]

        neighbor_wells = (
            training_well_names[
                selected_order
            ]
        )

        neighbor_distances = (
            distances[
                selected_order
            ]
        )

        adaptive_bandwidth = max(
            float(
                neighbor_distances[
                    -1
                ]
            ),
            bank[
                "median_nearest_distance"
            ],
            PROBABILITY_EPSILON,
        )

        neighbor_weights = np.exp(
            -0.5
            * (
                neighbor_distances
                / adaptive_bandwidth
            )
            ** 2
        )

        nearest_distance = float(
            neighbor_distances[
                0
            ]
        )

        distance_factor = float(
            np.exp(
                -0.5
                * (
                    nearest_distance
                    / bank[
                        "median_nearest_distance"
                    ]
                )
                ** 2
            )
        )

        query_z_all = query.loc[
            positions,
            vertical_column,
        ].to_numpy(
            dtype=float
        )

        valid_z_mask = np.isfinite(
            query_z_all
        )

        if not valid_z_mask.any():
            continue

        valid_positions = positions[
            valid_z_mask
        ]

        valid_query_z = query_z_all[
            valid_z_mask
        ]

        if (
            bank[
                "formation_available"
            ]
            and "FORMATION" in (
                query.columns
            )
        ):
            formation_values = (
                query.loc[
                    valid_positions,
                    "FORMATION",
                ]
            )

            formation_keys = np.where(
                formation_values.notna(),
                formation_values.astype(
                    str
                ),
                "__MISSING__",
            )

        else:
            formation_keys = np.full(
                len(valid_positions),
                "__NO_FORMATION__",
                dtype=object,
            )

        for formation_key in pd.unique(
            formation_keys
        ):
            local_selector = (
                formation_keys
                == formation_key
            )

            local_positions = (
                valid_positions[
                    local_selector
                ]
            )

            local_query_z = (
                valid_query_z[
                    local_selector
                ]
            )

            selected_result = None
            formation_rows_used = (
                np.zeros(
                    len(local_positions),
                    dtype=bool,
                )
            )

            use_formation = bool(
                bank[
                    "formation_available"
                ]
                and formation_key
                not in {
                    "__MISSING__",
                    "__NO_FORMATION__",
                }
            )

            if use_formation:
                formation_lookup = {
                    well_name: bank[
                        "formation_profiles"
                    ].get(
                        (
                            well_name,
                            str(
                                formation_key
                            ),
                        )
                    )
                    for well_name in (
                        neighbor_wells
                    )
                }

                formation_result = (
                    aggregate_neighbor_profiles(
                        local_query_z,
                        neighbor_wells,
                        neighbor_weights,
                        formation_lookup,
                        vertical_half_window,
                    )
                )

                formation_usable = (
                    formation_result[
                        "support_wells"
                    ]
                    >= MIN_FORMATION_SUPPORT_WELLS
                )

                if formation_usable.any():
                    all_result = (
                        aggregate_neighbor_profiles(
                            local_query_z,
                            neighbor_wells,
                            neighbor_weights,
                            bank[
                                "all_profiles"
                            ],
                            vertical_half_window,
                        )
                    )

                    for key in [
                        "aggregate",
                        "valid",
                        "support_wells",
                        "average_rows",
                        "agreement",
                    ]:
                        all_result[
                            key
                        ][
                            formation_usable
                        ] = formation_result[
                            key
                        ][
                            formation_usable
                        ]

                    selected_result = (
                        all_result
                    )

                    formation_rows_used = (
                        formation_usable
                    )

            if selected_result is None:
                selected_result = (
                    aggregate_neighbor_profiles(
                        local_query_z,
                        neighbor_wells,
                        neighbor_weights,
                        bank[
                            "all_profiles"
                        ],
                        vertical_half_window,
                    )
                )

            valid_local = selected_result[
                "valid"
            ]

            if not valid_local.any():
                continue

            local_prior = selected_result[
                "aggregate"
            ]

            support_wells = selected_result[
                "support_wells"
            ].astype(
                np.float64
            )

            smoothing = float(
                PRIOR_SMOOTHING_EQUIVALENT_WELLS
            )

            smoothed_prior = (
                support_wells[
                    :,
                    None,
                ]
                * local_prior
                + smoothing
                * bank[
                    "global_well_balanced_prior"
                ][
                    None,
                    :
                ]
            ) / (
                support_wells[
                    :,
                    None,
                ]
                + smoothing
            )

            smoothed_prior = (
                normalize_probability_rows(
                    smoothed_prior
                )
            )

            support_factor = np.clip(
                support_wells
                / float(
                    MIN_WELLS_FULL_TRUST
                ),
                0.0,
                1.0,
            )

            depth_factor = np.clip(
                selected_result[
                    "average_rows"
                ]
                / float(
                    ROWS_PER_WELL_FULL_TRUST
                ),
                0.0,
                1.0,
            )

            agreement_factor = np.clip(
                selected_result[
                    "agreement"
                ],
                0.0,
                1.0,
            )

            local_reliability = (
                support_factor
                * depth_factor
                * agreement_factor
                * distance_factor
            )

            prior_matrix[
                local_positions
            ] = smoothed_prior

            reliability_base[
                local_positions
            ] = local_reliability

            nearest_distance_output[
                local_positions
            ] = nearest_distance

            support_wells_output[
                local_positions
            ] = selected_result[
                "support_wells"
            ]

            average_rows_output[
                local_positions
            ] = selected_result[
                "average_rows"
            ]

            agreement_output[
                local_positions
            ] = selected_result[
                "agreement"
            ]

            formation_specific_output[
                local_positions
            ] = formation_rows_used

    return {
        "prior": prior_matrix,
        "reliability_base": (
            reliability_base
        ),
        "nearest_distance": (
            nearest_distance_output
        ),
        "support_wells": (
            support_wells_output
        ),
        "average_rows": (
            average_rows_output
        ),
        "agreement": (
            agreement_output
        ),
        "formation_specific": (
            formation_specific_output
        ),
    }

# 11. Support-aware stacking features and meta-model

In [19]:
def build_meta_features(
    legacy_probabilities,
    tcn_probabilities,
    geo_result,
    geo_bank,
):
    legacy_probabilities = (
        normalize_probability_rows(
            legacy_probabilities
        )
    )

    tcn_probabilities = (
        normalize_probability_rows(
            tcn_probabilities
        )
    )

    geo_prior = (
        normalize_probability_rows(
            geo_result[
                "prior"
            ]
        )
    )

    geo_global_prior = np.clip(
        geo_bank[
            "global_well_balanced_prior"
        ],
        PROBABILITY_EPSILON,
        1.0,
    )

    geo_log_ratio = (
        safe_log_probabilities(
            geo_prior
        )
        - np.log(
            geo_global_prior[
                None,
                :
            ]
        )
    )

    legacy_entropy = normalized_entropy(
        legacy_probabilities
    )

    tcn_entropy = normalized_entropy(
        tcn_probabilities
    )

    legacy_margin = probability_margin(
        legacy_probabilities
    )

    tcn_margin = probability_margin(
        tcn_probabilities
    )

    legacy_max = legacy_probabilities.max(
        axis=1
    )

    tcn_max = tcn_probabilities.max(
        axis=1
    )

    disagreement = (
        legacy_probabilities.argmax(
            axis=1
        )
        != tcn_probabilities.argmax(
            axis=1
        )
    ).astype(
        np.float64
    )

    nearest_distance = np.asarray(
        geo_result[
            "nearest_distance"
        ],
        dtype=np.float64,
    )

    normalized_distance = (
        nearest_distance
        / max(
            float(
                geo_bank[
                    "median_nearest_distance"
                ]
            ),
            PROBABILITY_EPSILON,
        )
    )

    normalized_distance = np.nan_to_num(
        normalized_distance,
        nan=10.0,
        posinf=10.0,
        neginf=10.0,
    )

    normalized_distance = np.clip(
        normalized_distance,
        0.0,
        10.0,
    )

    support_wells = np.asarray(
        geo_result[
            "support_wells"
        ],
        dtype=np.float64,
    )

    average_rows = np.asarray(
        geo_result[
            "average_rows"
        ],
        dtype=np.float64,
    )

    agreement = np.asarray(
        geo_result[
            "agreement"
        ],
        dtype=np.float64,
    )

    reliability = np.asarray(
        geo_result[
            "reliability_base"
        ],
        dtype=np.float64,
    )

    formation_specific = np.asarray(
        geo_result[
            "formation_specific"
        ],
        dtype=np.float64,
    )

    scalar_features = np.column_stack([
        legacy_entropy,
        tcn_entropy,
        legacy_margin,
        tcn_margin,
        legacy_max,
        tcn_max,
        legacy_max
        - tcn_max,
        legacy_entropy
        - tcn_entropy,
        disagreement,
        reliability,
        normalized_distance,
        support_wells
        / max(
            GEO_K_NEIGHBORS,
            1,
        ),
        average_rows
        / max(
            GEO_ROWS_PER_WELL_FULL_TRUST,
            1.0,
        ),
        agreement,
        formation_specific,
    ])

    feature_matrix = np.concatenate(
        [
            safe_log_probabilities(
                legacy_probabilities
            ),
            safe_log_probabilities(
                tcn_probabilities
            ),
            geo_log_ratio,
            scalar_features,
        ],
        axis=1,
    ).astype(
        np.float64
    )

    feature_names = (
        [
            f"legacy_logp_{code}"
            for code in CLASS_CODES
        ]
        + [
            f"tcn_logp_{code}"
            for code in CLASS_CODES
        ]
        + [
            f"geo_log_ratio_{code}"
            for code in CLASS_CODES
        ]
        + [
            "legacy_entropy",
            "tcn_entropy",
            "legacy_margin",
            "tcn_margin",
            "legacy_max_probability",
            "tcn_max_probability",
            "legacy_minus_tcn_max_probability",
            "legacy_minus_tcn_entropy",
            "expert_disagreement",
            "geo_reliability",
            "normalized_nearest_distance",
            "normalized_support_wells",
            "normalized_depth_rows",
            "geo_agreement",
            "formation_specific",
        ]
    )

    if feature_matrix.shape[
        1
    ] != len(
        feature_names
    ):
        raise RuntimeError(
            "Meta feature-name mismatch."
        )

    return {
        "X": feature_matrix,
        "feature_names": (
            feature_names
        ),
    }


def train_support_aware_meta_model(
    meta_features,
    metadata,
    random_state,
):
    selected_rows = subsample_meta_rows(
        metadata,
        random_state=random_state,
    )

    X_selected = meta_features[
        selected_rows
    ]

    y_selected = metadata.loc[
        selected_rows,
        "LITHOLOGY_ENCODED",
    ].to_numpy(
        dtype=int
    )

    sample_weights = (
        well_balanced_sample_weights(
            metadata.loc[
                selected_rows
            ].reset_index(
                drop=True
            )
        )
    )

    meta_scaler = StandardScaler()

    X_selected_scaled = (
        meta_scaler.fit_transform(
            X_selected
        )
    )

    meta_model = LogisticRegression(
        C=META_LOGISTIC_C,
        max_iter=META_MAX_ITER,
        solver="lbfgs",
        class_weight="balanced",
        random_state=(
            random_state
        ),
    )

    meta_model.fit(
        X_selected_scaled,
        y_selected,
        sample_weight=(
            sample_weights
        ),
    )

    return {
        "model": meta_model,
        "scaler": meta_scaler,
        "selected_rows": (
            selected_rows
        ),
    }


def predict_support_aware_meta_model(
    meta_bundle,
    meta_features,
):
    transformed = (
        meta_bundle[
            "scaler"
        ]
        .transform(
            meta_features
        )
    )

    partial_probabilities = (
        meta_bundle[
            "model"
        ]
        .predict_proba(
            transformed
        )
    )

    return align_probabilities(
        partial_probabilities,
        meta_bundle[
            "model"
        ].classes_,
    )


def calculate_generic_per_well_metrics(
    metadata,
    probabilities,
    predictions,
    variant,
    outer_fold,
):
    rows = []

    for well_name, row_indices in (
        metadata.groupby(
            "WELL"
        ).groups.items()
    ):
        positions = np.asarray(
            list(
                row_indices
            ),
            dtype=int,
        )

        y_true = metadata.loc[
            positions,
            "LITHOLOGY_ENCODED",
        ].to_numpy(
            dtype=int
        )

        present_labels = sorted(
            np.unique(
                y_true
            ).tolist()
        )

        rows.append({
            "outer_fold": int(
                outer_fold
            ),
            "variant": variant,
            "WELL": well_name,
            "rows": int(
                len(
                    positions
                )
            ),
            "accuracy": float(
                accuracy_score(
                    y_true,
                    predictions[
                        positions
                    ],
                )
            ),
            "balanced_accuracy": float(
                balanced_accuracy_score(
                    y_true,
                    predictions[
                        positions
                    ],
                )
            ),
            "macro_f1_present_classes": float(
                f1_score(
                    y_true,
                    predictions[
                        positions
                    ],
                    labels=present_labels,
                    average="macro",
                    zero_division=0,
                )
            ),
            "log_loss": float(
                log_loss(
                    y_true,
                    probabilities[
                        positions
                    ],
                    labels=CLASS_INDICES,
                )
            ),
        })

    return pd.DataFrame(
        rows
    )

# 12. Experiment signature and cache helpers

In [20]:
EXPERIMENT_CONFIGURATION = {
    "window_length": WINDOW_LENGTH,
    "TCN_filters": TCN_FILTERS,
    "TCN_dilations": TCN_DILATIONS,
    "TCN_kernel_size": TCN_KERNEL_SIZE,
    "TCN_dropout": TCN_DROPOUT,
    "TCN_dense_units": TCN_DENSE_UNITS,
    "TCN_features": TCN_FEATURES,
    "legacy_RF_parameters": (
        LEGACY_RF_PARAMETERS
    ),
    "geo_K": GEO_K_NEIGHBORS,
    "geo_vertical_half_window": (
        GEO_VERTICAL_HALF_WINDOW
    ),
    "meta_logistic_C": (
        META_LOGISTIC_C
    ),
    "primary_variant": (
        PRIMARY_VARIANT
    ),
    "official_test_used": False,
}

EXPERIMENT_SIGNATURE = hashlib.sha256(
    json.dumps(
        json_safe(
            EXPERIMENT_CONFIGURATION
        ),
        sort_keys=True,
    ).encode(
        "utf-8"
    )
).hexdigest()[
    :12
]

print(
    "Experiment signature:",
    EXPERIMENT_SIGNATURE,
)


def fold_cache_paths(
    outer_fold,
):
    prefix = (
        f"{EXPERIMENT_SIGNATURE}"
        f"__outer_{outer_fold}"
    )

    return {
        "inner_tcn": (
            CACHE_DIR
            / f"{prefix}__inner_tcn.npz"
        ),
        "inner_legacy": (
            CACHE_DIR
            / f"{prefix}__inner_legacy.npz"
        ),
        "outer_tcn": (
            CACHE_DIR
            / f"{prefix}__outer_tcn.npz"
        ),
        "meta_model": (
            CACHE_DIR
            / f"{prefix}__meta.joblib"
        ),
        "fold_output": (
            OUTPUT_DIR
            / f"{prefix}__fold_predictions.parquet"
        ),
        "fold_gate": (
            OUTPUT_DIR
            / f"{prefix}__fold_gate.json"
        ),
    }


def save_probability_cache(
    path,
    probabilities,
    original_row_ids,
    **metadata,
):
    np.savez_compressed(
        path,
        probabilities=np.asarray(
            probabilities,
            dtype=np.float32,
        ),
        original_row_ids=np.asarray(
            original_row_ids,
            dtype=np.int64,
        ),
        metadata_json=np.array(
            json.dumps(
                json_safe(
                    metadata
                )
            )
        ),
    )


def load_probability_cache(
    path,
    expected_row_ids,
):
    loaded = np.load(
        path,
        allow_pickle=False,
    )

    cached_row_ids = loaded[
        "original_row_ids"
    ].astype(
        np.int64
    )

    expected_row_ids = np.asarray(
        expected_row_ids,
        dtype=np.int64,
    )

    if not np.array_equal(
        cached_row_ids,
        expected_row_ids,
    ):
        raise RuntimeError(
            "Cached probability rows do not match current split."
        )

    metadata = json.loads(
        loaded[
            "metadata_json"
        ].item()
    )

    return {
        "probabilities": (
            loaded[
                "probabilities"
            ].astype(
                np.float64
            )
        ),
        "metadata": metadata,
    }

Experiment signature: 19504b11c4e4


# 13. Load Train and grouped WELL folds

In [21]:
with open(
    LEGACY_REFERENCE_GATE_PATH,
    "r",
    encoding="utf-8",
) as file:
    legacy_reference_gate = (
        json.load(
            file
        )
    )

if not legacy_reference_gate.get(
    "overall_reproduction_passed",
    False,
):
    raise RuntimeError(
        "Exact Legacy reproduction gate did not pass."
    )

train_raw = load_csv_full(
    TRAIN_PATH
)

required_columns = {
    "WELL",
    "DEPTH_MD",
    TARGET_COLUMN,
    "X_LOC",
    "Y_LOC",
}

missing_required_columns = (
    required_columns
    - set(
        train_raw.columns
    )
)

if missing_required_columns:
    raise KeyError(
        "Missing required Train columns: "
        + str(
            sorted(
                missing_required_columns
            )
        )
    )

train_raw[
    "ORIGINAL_ROW_ID"
] = np.arange(
    len(
        train_raw
    ),
    dtype=np.int64,
)

train_raw[
    TARGET_COLUMN
] = pd.to_numeric(
    train_raw[
        TARGET_COLUMN
    ],
    errors="raise",
).astype(
    int
)

fold_assignment_raw = pd.read_csv(
    FOLD_ASSIGNMENT_PATH
)

well_candidates = [
    column
    for column in (
        fold_assignment_raw.columns
    )
    if column.upper()
    == "WELL"
]

fold_candidates = [
    column
    for column in (
        fold_assignment_raw.columns
    )
    if (
        "FOLD"
        in column.upper()
        and "SCORE"
        not in column.upper()
    )
]

if (
    not well_candidates
    or not fold_candidates
):
    raise RuntimeError(
        "Could not infer WELL/FOLD columns."
    )

fold_assignment = (
    fold_assignment_raw[
        [
            well_candidates[
                0
            ],
            fold_candidates[
                0
            ],
        ]
    ]
    .rename(
        columns={
            well_candidates[
                0
            ]: "WELL",
            fold_candidates[
                0
            ]: "GROUP_FOLD",
        }
    )
    .drop_duplicates(
        subset=[
            "WELL"
        ]
    )
)

fold_assignment[
    "GROUP_FOLD"
] = pd.to_numeric(
    fold_assignment[
        "GROUP_FOLD"
    ],
    errors="raise",
).astype(
    int
)

train_raw[
    "GROUP_FOLD"
] = train_raw[
    "WELL"
].map(
    dict(
        zip(
            fold_assignment[
                "WELL"
            ],
            fold_assignment[
                "GROUP_FOLD"
            ],
        )
    )
)

if train_raw[
    "GROUP_FOLD"
].isna().any():
    missing_wells = (
        train_raw.loc[
            train_raw[
                "GROUP_FOLD"
            ].isna(),
            "WELL",
        ]
        .dropna()
        .unique()
        .tolist()
    )

    raise RuntimeError(
        "Wells without fold assignment: "
        + str(
            missing_wells[
                :20
            ]
        )
    )

train_raw[
    "GROUP_FOLD"
] = train_raw[
    "GROUP_FOLD"
].astype(
    int
)

observed_folds = sorted(
    train_raw[
        "GROUP_FOLD"
    ].unique()
)

if observed_folds != [
    1,
    2,
    3,
    4,
    5,
]:
    raise RuntimeError(
        "Expected five grouped WELL folds, observed: "
        + str(
            observed_folds
        )
    )

print("Train shape:", train_raw.shape)
print(
    "Train wells:",
    train_raw[
        "WELL"
    ].nunique(),
)
print(
    "Fold row counts:"
)
display(
    train_raw[
        "GROUP_FOLD"
    ].value_counts(
        sort=False
    ).sort_index()
)

Loading train.csv with {'sep': ',', 'encoding': 'utf-8'}
Train shape: (1170511, 31)
Train wells: 98
Fold row counts:


GROUP_FOLD
1    232674
2    236205
3    230578
4    237132
5    233922
Name: count, dtype: int64

In [22]:
def choose_inner_gate_fold(
    outer_fold,
):
    candidate = (
        outer_fold
        % 5
    ) + 1

    if candidate == outer_fold:
        raise RuntimeError(
            "Inner gate fold equals outer fold."
        )

    return int(
        candidate
    )


split_registry_rows = []

for outer_fold in [
    1,
    2,
    3,
    4,
    5,
]:
    gate_fold = (
        choose_inner_gate_fold(
            outer_fold
        )
    )

    development_folds = [
        fold
        for fold in [
            1,
            2,
            3,
            4,
            5,
        ]
        if fold not in {
            outer_fold,
            gate_fold,
        }
    ]

    split_registry_rows.append({
        "outer_fold": (
            outer_fold
        ),
        "inner_gate_fold": (
            gate_fold
        ),
        "inner_development_folds": (
            ",".join(
                map(
                    str,
                    development_folds,
                )
            )
        ),
    })

split_registry = pd.DataFrame(
    split_registry_rows
)

display(
    split_registry
)

,outer_fold,inner_gate_fold,inner_development_folds
0,1,2,"3,4,5"
1,2,3,"1,4,5"
2,3,4,"1,2,5"
3,4,5,"1,2,3"
4,5,1,"2,3,4"


# 14. Nested grouped-WELL stacking experiment

In [23]:
def make_metadata_frame(
    raw_frame,
):
    metadata = (
        raw_frame.copy()
        .reset_index(
            drop=True
        )
    )

    metadata[
        TARGET_COLUMN
    ] = pd.to_numeric(
        metadata[
            TARGET_COLUMN
        ],
        errors="raise",
    ).astype(
        int
    )

    metadata[
        "LITHOLOGY_ENCODED"
    ] = label_encoder.transform(
        metadata[
            TARGET_COLUMN
        ]
    )

    keep_columns = [
        "ORIGINAL_ROW_ID",
        "WELL",
        "DEPTH_MD",
        TARGET_COLUMN,
        "LITHOLOGY_ENCODED",
    ]

    for optional_column in [
        "X_LOC",
        "Y_LOC",
        "Z_LOC",
        "FORMATION",
    ]:
        if optional_column in (
            metadata.columns
        ):
            keep_columns.append(
                optional_column
            )

    return (
        metadata[
            keep_columns
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

if RUN_MODE in {
    "pilot",
    "full_cv",
}:
    all_fold_metric_rows = []
    all_per_class_frames = []
    all_per_well_frames = []
    all_prediction_frames = []
    all_geo_diagnostics = []
    all_fold_gate_rows = []

    for outer_fold in ACTIVE_OUTER_FOLDS:
        fold_start_time = time.time()

        print(
            "\\n"
            + "=" * 120
        )

        print(
            "OUTER FOLD",
            outer_fold,
        )

        print(
            "=" * 120
        )

        paths = fold_cache_paths(
            outer_fold
        )

        inner_gate_fold = (
            choose_inner_gate_fold(
                outer_fold
            )
        )

        outer_validation_mask = (
            train_raw[
                "GROUP_FOLD"
            ]
            == outer_fold
        )

        inner_gate_mask = (
            train_raw[
                "GROUP_FOLD"
            ]
            == inner_gate_fold
        )

        inner_development_mask = (
            ~outer_validation_mask
            & ~inner_gate_mask
        )

        outer_training_mask = (
            ~outer_validation_mask
        )

        raw_inner_development = (
            train_raw.loc[
                inner_development_mask
            ]
            .drop(
                columns=[
                    "GROUP_FOLD"
                ]
            )
            .copy()
            .reset_index(
                drop=True
            )
        )

        raw_inner_gate = (
            train_raw.loc[
                inner_gate_mask
            ]
            .drop(
                columns=[
                    "GROUP_FOLD"
                ]
            )
            .copy()
            .reset_index(
                drop=True
            )
        )

        raw_outer_train = (
            train_raw.loc[
                outer_training_mask
            ]
            .drop(
                columns=[
                    "GROUP_FOLD"
                ]
            )
            .copy()
            .reset_index(
                drop=True
            )
        )

        raw_outer_validation = (
            train_raw.loc[
                outer_validation_mask
            ]
            .drop(
                columns=[
                    "GROUP_FOLD"
                ]
            )
            .copy()
            .reset_index(
                drop=True
            )
        )

        development_wells = set(
            raw_inner_development[
                "WELL"
            ]
        )

        inner_gate_wells = set(
            raw_inner_gate[
                "WELL"
            ]
        )

        outer_validation_wells = set(
            raw_outer_validation[
                "WELL"
            ]
        )

        outer_train_wells = set(
            raw_outer_train[
                "WELL"
            ]
        )

        if (
            development_wells
            & inner_gate_wells
        ):
            raise RuntimeError(
                "Inner development/gate WELL leakage."
            )

        if (
            outer_train_wells
            & outer_validation_wells
        ):
            raise RuntimeError(
                "Outer Train/Validation WELL leakage."
            )

        inner_gate_metadata = (
            make_metadata_frame(
                raw_inner_gate
            )
        )

        outer_validation_metadata = (
            make_metadata_frame(
                raw_outer_validation
            )
        )

        # --------------------------------------------------------
        # Inner TCN: trained only on three development folds.
        # The gate fold is never used for training or epoch tuning.
        # --------------------------------------------------------

        if paths[
            "inner_tcn"
        ].exists():
            inner_tcn_cache = (
                load_probability_cache(
                    paths[
                        "inner_tcn"
                    ],
                    inner_gate_metadata[
                        "ORIGINAL_ROW_ID"
                    ].to_numpy(
                        dtype=np.int64
                    ),
                )
            )

            inner_tcn_probabilities = (
                inner_tcn_cache[
                    "probabilities"
                ]
            )

            fixed_tcn_epochs = int(
                inner_tcn_cache[
                    "metadata"
                ][
                    "fixed_tcn_epochs"
                ]
            )

            print(
                "Loaded cached inner TCN."
            )

        else:
            inner_tcn_result = (
                train_inner_tcn(
                    raw_inner_development,
                    raw_inner_gate,
                    outer_fold=(
                        outer_fold
                    ),
                )
            )

            if not np.array_equal(
                inner_tcn_result[
                    "gate_frame"
                ][
                    "ORIGINAL_ROW_ID"
                ].to_numpy(
                    dtype=np.int64
                ),
                inner_gate_metadata[
                    "ORIGINAL_ROW_ID"
                ].to_numpy(
                    dtype=np.int64
                ),
            ):
                raise RuntimeError(
                    "Inner TCN prediction order mismatch."
                )

            inner_tcn_probabilities = (
                inner_tcn_result[
                    "gate_probabilities"
                ]
            )

            fixed_tcn_epochs = int(
                inner_tcn_result[
                    "best_epoch"
                ]
            )

            save_probability_cache(
                paths[
                    "inner_tcn"
                ],
                inner_tcn_probabilities,
                inner_gate_metadata[
                    "ORIGINAL_ROW_ID"
                ].to_numpy(
                    dtype=np.int64
                ),
                fixed_tcn_epochs=(
                    fixed_tcn_epochs
                ),
                gate_labels_used_for_training_or_selection=False,
            )

        # --------------------------------------------------------
        # Inner exact Legacy RF.
        # --------------------------------------------------------

        if paths[
            "inner_legacy"
        ].exists():
            inner_legacy_cache = (
                load_probability_cache(
                    paths[
                        "inner_legacy"
                    ],
                    inner_gate_metadata[
                        "ORIGINAL_ROW_ID"
                    ].to_numpy(
                        dtype=np.int64
                    ),
                )
            )

            inner_legacy_probabilities = (
                inner_legacy_cache[
                    "probabilities"
                ]
            )

            print(
                "Loaded cached inner Legacy RF."
            )

        else:
            inner_legacy_result = (
                train_exact_legacy_full_and_predict(
                    raw_inner_development,
                    raw_inner_gate,
                )
            )

            if not np.array_equal(
                inner_legacy_result[
                    "metadata"
                ][
                    "ORIGINAL_ROW_ID"
                ].to_numpy(
                    dtype=np.int64
                ),
                inner_gate_metadata[
                    "ORIGINAL_ROW_ID"
                ].to_numpy(
                    dtype=np.int64
                ),
            ):
                raise RuntimeError(
                    "Inner Legacy prediction order mismatch."
                )

            inner_legacy_probabilities = (
                inner_legacy_result[
                    "probabilities"
                ]
            )

            save_probability_cache(
                paths[
                    "inner_legacy"
                ],
                inner_legacy_probabilities,
                inner_gate_metadata[
                    "ORIGINAL_ROW_ID"
                ].to_numpy(
                    dtype=np.int64
                ),
                exact_legacy_full_RF=True,
            )

        # --------------------------------------------------------
        # Inner geo context and nested meta-model.
        # --------------------------------------------------------

        inner_geo_bank = build_geo_prior_bank(
            raw_inner_development
        )

        inner_geo_result = compute_geo_prior(
            raw_inner_gate,
            inner_geo_bank,
            k_neighbors=(
                GEO_K_NEIGHBORS
            ),
            vertical_half_window=(
                GEO_VERTICAL_HALF_WINDOW
            ),
        )

        inner_meta_features = (
            build_meta_features(
                legacy_probabilities=(
                    inner_legacy_probabilities
                ),
                tcn_probabilities=(
                    inner_tcn_probabilities
                ),
                geo_result=(
                    inner_geo_result
                ),
                geo_bank=(
                    inner_geo_bank
                ),
            )
        )

        if paths[
            "meta_model"
        ].exists():
            meta_bundle = joblib.load(
                paths[
                    "meta_model"
                ]
            )

            print(
                "Loaded cached support-aware meta-model."
            )

        else:
            meta_bundle = (
                train_support_aware_meta_model(
                    inner_meta_features[
                        "X"
                    ],
                    inner_gate_metadata,
                    random_state=(
                        RANDOM_STATE
                        + outer_fold
                    ),
                )
            )

            meta_bundle[
                "feature_names"
            ] = inner_meta_features[
                "feature_names"
            ]

            joblib.dump(
                meta_bundle,
                paths[
                    "meta_model"
                ],
                compress=3,
            )

            coefficient_frame = pd.DataFrame(
                meta_bundle[
                    "model"
                ].coef_,
                columns=(
                    inner_meta_features[
                        "feature_names"
                    ]
                ),
            )

            coefficient_frame[
                "trained_class_index"
            ] = (
                meta_bundle[
                    "model"
                ].classes_
            )

            coefficient_frame.to_csv(
                OUTPUT_DIR
                / (
                    f"outer_{outer_fold}"
                    "__meta_coefficients.csv"
                ),
                index=False,
            )

        # --------------------------------------------------------
        # Final TCN for this outer fold.
        # It is retrained on all four outer-training folds.
        # --------------------------------------------------------

        if paths[
            "outer_tcn"
        ].exists():
            outer_tcn_cache = (
                load_probability_cache(
                    paths[
                        "outer_tcn"
                    ],
                    outer_validation_metadata[
                        "ORIGINAL_ROW_ID"
                    ].to_numpy(
                        dtype=np.int64
                    ),
                )
            )

            outer_tcn_probabilities = (
                outer_tcn_cache[
                    "probabilities"
                ]
            )

            print(
                "Loaded cached outer TCN."
            )

        else:
            outer_tcn_result = (
                train_outer_tcn_and_predict(
                    raw_outer_train,
                    raw_outer_validation,
                    best_epoch=(
                        fixed_tcn_epochs
                    ),
                    outer_fold=(
                        outer_fold
                    ),
                )
            )

            if not np.array_equal(
                outer_tcn_result[
                    "frame"
                ][
                    "ORIGINAL_ROW_ID"
                ].to_numpy(
                    dtype=np.int64
                ),
                outer_validation_metadata[
                    "ORIGINAL_ROW_ID"
                ].to_numpy(
                    dtype=np.int64
                ),
            ):
                raise RuntimeError(
                    "Outer TCN prediction order mismatch."
                )

            outer_tcn_probabilities = (
                outer_tcn_result[
                    "probabilities"
                ]
            )

            save_probability_cache(
                paths[
                    "outer_tcn"
                ],
                outer_tcn_probabilities,
                outer_validation_metadata[
                    "ORIGINAL_ROW_ID"
                ].to_numpy(
                    dtype=np.int64
                ),
                fixed_tcn_epochs=(
                    fixed_tcn_epochs
                ),
                weights_path=(
                    outer_tcn_result[
                        "weights_path"
                    ]
                ),
            )

        # Exact Legacy outer probabilities are reused from 02B.
        outer_legacy_result = (
            load_legacy_outer_oof(
                outer_fold,
                raw_outer_validation,
            )
        )

        outer_legacy_probabilities = (
            outer_legacy_result[
                "probabilities"
            ]
        )

        # Geo context is built only from outer-training wells.
        outer_geo_bank = build_geo_prior_bank(
            raw_outer_train
        )

        outer_geo_result = compute_geo_prior(
            raw_outer_validation,
            outer_geo_bank,
            k_neighbors=(
                GEO_K_NEIGHBORS
            ),
            vertical_half_window=(
                GEO_VERTICAL_HALF_WINDOW
            ),
        )

        outer_meta_features = (
            build_meta_features(
                legacy_probabilities=(
                    outer_legacy_probabilities
                ),
                tcn_probabilities=(
                    outer_tcn_probabilities
                ),
                geo_result=(
                    outer_geo_result
                ),
                geo_bank=(
                    outer_geo_bank
                ),
            )
        )

        stacked_probabilities = (
            predict_support_aware_meta_model(
                meta_bundle,
                outer_meta_features[
                    "X"
                ],
            )
        )

        simple_average_probabilities = (
            normalize_probability_rows(
                0.5
                * outer_legacy_probabilities
                + 0.5
                * outer_tcn_probabilities
            )
        )

        variant_probabilities = {
            "legacy_only": (
                outer_legacy_probabilities
            ),
            "tcn_only": (
                outer_tcn_probabilities
            ),
            "simple_average": (
                simple_average_probabilities
            ),
            "support_aware_stacking": (
                stacked_probabilities
            ),
        }

        y_outer = (
            outer_validation_metadata[
                "LITHOLOGY_ENCODED"
            ].to_numpy(
                dtype=int
            )
        )

        prediction_output = (
            outer_validation_metadata.copy()
        )

        prediction_output[
            "outer_fold"
        ] = int(
            outer_fold
        )

        prediction_output[
            "inner_gate_fold"
        ] = int(
            inner_gate_fold
        )

        prediction_output[
            "geo_reliability"
        ] = (
            outer_geo_result[
                "reliability_base"
            ]
        )

        prediction_output[
            "geo_nearest_distance"
        ] = (
            outer_geo_result[
                "nearest_distance"
            ]
        )

        prediction_output[
            "geo_support_wells"
        ] = (
            outer_geo_result[
                "support_wells"
            ]
        )

        prediction_output[
            "geo_average_rows"
        ] = (
            outer_geo_result[
                "average_rows"
            ]
        )

        prediction_output[
            "geo_agreement"
        ] = (
            outer_geo_result[
                "agreement"
            ]
        )

        prediction_output[
            "geo_formation_specific"
        ] = (
            outer_geo_result[
                "formation_specific"
            ]
        )

        fold_probability_checks = {}

        for variant, probabilities in (
            variant_probabilities.items()
        ):
            probabilities = (
                normalize_probability_rows(
                    probabilities
                )
            )

            predictions = probabilities.argmax(
                axis=1
            ).astype(
                int
            )

            metrics = calculate_metrics(
                y_outer,
                predictions,
                probabilities=(
                    probabilities
                ),
            )

            all_fold_metric_rows.append({
                "outer_fold": int(
                    outer_fold
                ),
                "inner_gate_fold": int(
                    inner_gate_fold
                ),
                "variant": variant,
                "is_primary_variant": (
                    variant
                    == PRIMARY_VARIANT
                ),
                "fixed_tcn_epochs": int(
                    fixed_tcn_epochs
                ),
                **metrics,
            })

            per_class = (
                calculate_per_class_metrics(
                    y_outer,
                    predictions,
                )
            )

            per_class[
                "outer_fold"
            ] = int(
                outer_fold
            )

            per_class[
                "variant"
            ] = variant

            all_per_class_frames.append(
                per_class
            )

            per_well = (
                calculate_generic_per_well_metrics(
                    outer_validation_metadata,
                    probabilities,
                    predictions,
                    variant=variant,
                    outer_fold=(
                        outer_fold
                    ),
                )
            )

            all_per_well_frames.append(
                per_well
            )

            prediction_output[
                f"{variant}__prediction_index"
            ] = predictions

            for class_index, class_code in enumerate(
                CLASS_CODES
            ):
                prediction_output[
                    (
                        f"{variant}"
                        f"__prob_{class_code}"
                    )
                ] = probabilities[
                    :,
                    class_index,
                ].astype(
                    np.float32
                )

            fold_probability_checks[
                variant
            ] = {
                "finite": bool(
                    np.isfinite(
                        probabilities
                    ).all()
                ),
                "sum_to_one": bool(
                    np.allclose(
                        probabilities.sum(
                            axis=1
                        ),
                        1.0,
                        rtol=1e-6,
                        atol=1e-6,
                    )
                ),
            }

        prediction_output.to_parquet(
            paths[
                "fold_output"
            ],
            index=False,
        )

        all_prediction_frames.append(
            prediction_output
        )

        all_geo_diagnostics.append({
            "outer_fold": int(
                outer_fold
            ),
            "vertical_column": (
                outer_geo_bank[
                    "vertical_column"
                ]
            ),
            "mean_reliability": float(
                outer_geo_result[
                    "reliability_base"
                ].mean()
            ),
            "zero_reliability_fraction": float(
                (
                    outer_geo_result[
                        "reliability_base"
                    ]
                    <= 0
                ).mean()
            ),
            "mean_support_wells": float(
                outer_geo_result[
                    "support_wells"
                ].mean()
            ),
            "mean_depth_rows": float(
                outer_geo_result[
                    "average_rows"
                ].mean()
            ),
            "mean_agreement": float(
                outer_geo_result[
                    "agreement"
                ].mean()
            ),
            "formation_specific_fraction": float(
                outer_geo_result[
                    "formation_specific"
                ].mean()
            ),
        })

        fold_gate = {
            "outer_fold": int(
                outer_fold
            ),
            "inner_gate_fold": int(
                inner_gate_fold
            ),
            "official_test_unused": True,
            "inner_development_gate_well_overlap_zero": (
                len(
                    development_wells
                    & inner_gate_wells
                )
                == 0
            ),
            "outer_train_validation_well_overlap_zero": (
                len(
                    outer_train_wells
                    & outer_validation_wells
                )
                == 0
            ),
            "inner_gate_labels_not_used_for_TCN_training_or_epoch_selection": True,
            "inner_legacy_trained_without_gate_wells": True,
            "outer_legacy_probabilities_from_exact_02B_OOF": True,
            "outer_TCN_has_no_raw_XY_features": True,
            "geo_context_uses_outer_training_labels_only": True,
            "meta_model_trained_only_on_inner_gate_fold": True,
            "probability_checks": (
                fold_probability_checks
            ),
            "fold_runtime_seconds": float(
                time.time()
                - fold_start_time
            ),
        }

        fold_gate[
            "fold_passed"
        ] = bool(
            fold_gate[
                "official_test_unused"
            ]
            and fold_gate[
                "inner_development_gate_well_overlap_zero"
            ]
            and fold_gate[
                "outer_train_validation_well_overlap_zero"
            ]
            and fold_gate[
                "inner_gate_labels_not_used_for_TCN_training_or_epoch_selection"
            ]
            and fold_gate[
                "inner_legacy_trained_without_gate_wells"
            ]
            and fold_gate[
                "outer_legacy_probabilities_from_exact_02B_OOF"
            ]
            and fold_gate[
                "outer_TCN_has_no_raw_XY_features"
            ]
            and fold_gate[
                "geo_context_uses_outer_training_labels_only"
            ]
            and fold_gate[
                "meta_model_trained_only_on_inner_gate_fold"
            ]
            and all(
                check[
                    "finite"
                ]
                and check[
                    "sum_to_one"
                ]
                for check in (
                    fold_probability_checks.values()
                )
            )
        )

        with open(
            paths[
                "fold_gate"
            ],
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(
                json_safe(
                    fold_gate
                ),
                file,
                ensure_ascii=False,
                indent=2,
            )

        all_fold_gate_rows.append(
            fold_gate
        )

        print(
            json.dumps(
                fold_gate,
                indent=2,
            )
        )

        if not fold_gate[
            "fold_passed"
        ]:
            raise RuntimeError(
                f"Outer fold {outer_fold} gate failed."
            )

        del inner_tcn_probabilities
        del inner_legacy_probabilities
        del inner_geo_bank
        del inner_geo_result
        del inner_meta_features
        del outer_tcn_probabilities
        del outer_legacy_probabilities
        del outer_geo_bank
        del outer_geo_result
        del outer_meta_features
        del stacked_probabilities
        del simple_average_probabilities
        gc.collect()

\n========================================================================================================================
OUTER FOLD 1


2026-08-01 19:26:44.414421: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2 Pro
2026-08-01 19:26:44.414491: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-08-01 19:26:44.414502: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.33 GB
2026-08-01 19:26:44.414555: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-08-01 19:26:44.414588: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Epoch 1/12


2026-08-01 19:26:46.226729: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2026-08-01 19:26:46.370457: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] PluggableGraphOptimizer failed: INVALID_ARGUMENT: Failed to deserialize the `graph_buf`.


120/120 [==============================] - 17s 104ms/step - loss: 1.3999 - sparse_categorical_accuracy: 0.3876
Epoch 2/12
120/120 [==============================] - 12s 101ms/step - loss: 0.9204 - sparse_categorical_accuracy: 0.5375
Epoch 3/12
120/120 [==============================] - 12s 102ms/step - loss: 0.7940 - sparse_categorical_accuracy: 0.5734
Epoch 4/12
120/120 [==============================] - 12s 102ms/step - loss: 0.7180 - sparse_categorical_accuracy: 0.6064
Epoch 5/12
120/120 [==============================] - 12s 100ms/step - loss: 0.6736 - sparse_categorical_accuracy: 0.6246
Epoch 6/12
120/120 [==============================] - 12s 102ms/step - loss: 0.6386 - sparse_categorical_accuracy: 0.6341
Epoch 7/12
120/120 [==============================] - 12s 101ms/step - loss: 0.6075 - sparse_categorical_accuracy: 0.6506
Epoch 8/12
120/120 [==============================] - 12s 102ms/step - loss: 0.5805 - sparse_categorical_accuracy: 0.6594
Epoch 9/12
120/120 [===============

...

Epoch 1/12


2026-08-01 19:35:41.041093: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] PluggableGraphOptimizer failed: INVALID_ARGUMENT: Failed to deserialize the `graph_buf`.


180/180 [==============================] - 22s 106ms/step - loss: 1.2128 - sparse_categorical_accuracy: 0.4298
Epoch 2/12
180/180 [==============================] - 18s 101ms/step - loss: 0.8346 - sparse_categorical_accuracy: 0.5534
Epoch 3/12
180/180 [==============================] - 18s 102ms/step - loss: 0.7215 - sparse_categorical_accuracy: 0.5903
Epoch 4/12
180/180 [==============================] - 18s 101ms/step - loss: 0.6607 - sparse_categorical_accuracy: 0.6118
Epoch 5/12
180/180 [==============================] - 18s 102ms/step - loss: 0.6239 - sparse_categorical_accuracy: 0.6291
Epoch 6/12
180/180 [==============================] - 18s 102ms/step - loss: 0.5833 - sparse_categorical_accuracy: 0.6507
Epoch 7/12
180/180 [==============================] - 18s 99ms/step - loss: 0.5673 - sparse_categorical_accuracy: 0.6531
Epoch 8/12
180/180 [==============================] - 18s 102ms/step - loss: 0.5400 - sparse_categorical_accuracy: 0.6646
Epoch 9/12
180/180 [================

2026-08-01 19:39:22.175765: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] PluggableGraphOptimizer failed: INVALID_ARGUMENT: Failed to deserialize the `graph_buf`.


455/455 [==============================] - 20s 43ms/step
{
  "outer_fold": 1,
  "inner_gate_fold": 2,
  "official_test_unused": true,
  "inner_development_gate_well_overlap_zero": true,
  "outer_train_validation_well_overlap_zero": true,
  "inner_gate_labels_not_used_for_TCN_training_or_epoch_selection": true,
  "inner_legacy_trained_without_gate_wells": true,
  "outer_legacy_probabilities_from_exact_02B_OOF": true,
  "outer_TCN_has_no_raw_XY_features": true,
  "geo_context_uses_outer_training_labels_only": true,
  "meta_model_trained_only_on_inner_gate_fold": true,
  "probability_checks": {
    "legacy_only": {
      "finite": true,
      "sum_to_one": true
    },
    "tcn_only": {
      "finite": true,
      "sum_to_one": true
    },
    "simple_average": {
      "finite": true,
      "sum_to_one": true
    },
    "support_aware_stacking": {
      "finite": true,
      "sum_to_one": true
    }
  },
  "fold_runtime_seconds": 786.6223380565643,
  "fold_passed": true
}


# 15. Aggregate grouped-CV results

In [24]:
if RUN_MODE in {
    "pilot",
    "full_cv",
}:
    fold_metrics = pd.DataFrame(
        all_fold_metric_rows
    )

    per_class_metrics = pd.concat(
        all_per_class_frames,
        ignore_index=True,
    )

    per_well_metrics = pd.concat(
        all_per_well_frames,
        ignore_index=True,
    )

    geo_diagnostics = pd.DataFrame(
        all_geo_diagnostics
    )

    oof_predictions = pd.concat(
        all_prediction_frames,
        ignore_index=True,
    ).sort_values(
        "ORIGINAL_ROW_ID"
    ).reset_index(
        drop=True
    )

    variant_summary_rows = []

    for variant in MODEL_VARIANTS:
        variant_wells = (
            per_well_metrics.loc[
                per_well_metrics[
                    "variant"
                ]
                == variant
            ]
        )

        probability_columns = [
            (
                f"{variant}"
                f"__prob_{class_code}"
            )
            for class_code in (
                CLASS_CODES
            )
        ]

        probabilities = (
            oof_predictions[
                probability_columns
            ]
            .to_numpy(
                dtype=np.float64
            )
        )

        predictions = (
            oof_predictions[
                f"{variant}__prediction_index"
            ]
            .to_numpy(
                dtype=int
            )
        )

        y_true = (
            oof_predictions[
                "LITHOLOGY_ENCODED"
            ]
            .to_numpy(
                dtype=int
            )
        )

        row_metrics = calculate_metrics(
            y_true,
            predictions,
            probabilities=probabilities,
        )

        variant_summary_rows.append({
            "variant": variant,
            "is_primary_variant": (
                variant
                == PRIMARY_VARIANT
            ),
            "outer_folds": int(
                fold_metrics.loc[
                    fold_metrics[
                        "variant"
                    ]
                    == variant,
                    "outer_fold",
                ].nunique()
            ),
            "rows": int(
                len(
                    y_true
                )
            ),
            "wells": int(
                variant_wells[
                    "WELL"
                ].nunique()
            ),
            "row_accuracy": (
                row_metrics[
                    "accuracy"
                ]
            ),
            "row_balanced_accuracy": (
                row_metrics[
                    "balanced_accuracy"
                ]
            ),
            "row_macro_f1_present_classes": (
                row_metrics[
                    "macro_f1_present_classes"
                ]
            ),
            "row_macro_f1_all_12_classes": (
                row_metrics[
                    "macro_f1_all_12_classes"
                ]
            ),
            "row_weighted_f1": (
                row_metrics[
                    "weighted_f1"
                ]
            ),
            "row_log_loss": (
                row_metrics[
                    "log_loss"
                ]
            ),
            "well_balanced_accuracy": float(
                variant_wells[
                    "accuracy"
                ].mean()
            ),
            "well_balanced_balanced_accuracy": float(
                variant_wells[
                    "balanced_accuracy"
                ].mean()
            ),
            "well_balanced_macro_f1": float(
                variant_wells[
                    "macro_f1_present_classes"
                ].mean()
            ),
            "well_balanced_log_loss": float(
                variant_wells[
                    "log_loss"
                ].mean()
            ),
            "worst_well_log_loss": float(
                variant_wells[
                    "log_loss"
                ].max()
            ),
        })

    variant_summary = pd.DataFrame(
        variant_summary_rows
    ).sort_values(
        [
            "well_balanced_macro_f1",
            "well_balanced_balanced_accuracy",
            "well_balanced_log_loss",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    ).reset_index(
        drop=True
    )

    legacy_reference = (
        variant_summary.loc[
            variant_summary[
                "variant"
            ]
            == "legacy_only"
        ]
        .iloc[
            0
        ]
    )

    variant_summary[
        "delta_well_macro_f1_vs_legacy"
    ] = (
        variant_summary[
            "well_balanced_macro_f1"
        ]
        - float(
            legacy_reference[
                "well_balanced_macro_f1"
            ]
        )
    )

    variant_summary[
        "delta_well_balanced_accuracy_vs_legacy"
    ] = (
        variant_summary[
            "well_balanced_balanced_accuracy"
        ]
        - float(
            legacy_reference[
                "well_balanced_balanced_accuracy"
            ]
        )
    )

    variant_summary[
        "delta_well_log_loss_vs_legacy"
    ] = (
        variant_summary[
            "well_balanced_log_loss"
        ]
        - float(
            legacy_reference[
                "well_balanced_log_loss"
            ]
        )
    )

    print(
        "Grouped-CV variant comparison:"
    )

    display(
        variant_summary
    )

Grouped-CV variant comparison:


,variant,is_primary_variant,outer_folds,rows,wells,row_accuracy,row_balanced_accuracy,row_macro_f1_present_classes,row_macro_f1_all_12_classes,row_weighted_f1,row_log_loss,well_balanced_accuracy,well_balanced_balanced_accuracy,well_balanced_macro_f1,well_balanced_log_loss,worst_well_log_loss,delta_well_macro_f1_vs_legacy,delta_well_balanced_accuracy_vs_legacy,delta_well_log_loss_vs_legacy
0,simple_average,False,1,232674,20,0.711884,0.558694,0.432916,0.396840,0.696991,0.858130,0.699736,0.462845,0.421557,0.867622,1.888101,0.048151,0.077554,0.044724
1,tcn_only,False,1,232674,20,0.614632,0.563803,0.368901,0.338159,0.642472,1.179361,0.612984,0.453476,0.389742,1.206764,2.775907,0.016336,0.068185,0.383866
2,legacy_only,False,1,232674,20,0.738222,0.364479,0.381822,0.350004,0.686007,0.811111,0.721910,0.385290,0.373406,0.822898,1.853161,0.000000,0.000000,0.000000
3,support_aware_stacking,True,1,232674,20,0.055086,0.142574,0.079747,0.073101,0.088316,16.547283,0.029949,0.045035,0.036861,17.357549,18.420681,-0.336545,-0.340255,16.534652


# 16. Save outputs and stage gate

In [25]:
if RUN_MODE in {
    "pilot",
    "full_cv",
}:
    fold_metrics.to_csv(
        OUTPUT_DIR
        / "01_fold_metrics.csv",
        index=False,
    )

    variant_summary.to_csv(
        OUTPUT_DIR
        / "02_variant_summary.csv",
        index=False,
    )

    per_class_metrics.to_csv(
        OUTPUT_DIR
        / "03_per_class_metrics.csv",
        index=False,
    )

    per_well_metrics.to_csv(
        OUTPUT_DIR
        / "04_per_well_metrics.csv",
        index=False,
    )

    geo_diagnostics.to_csv(
        OUTPUT_DIR
        / "05_geo_diagnostics.csv",
        index=False,
    )

    split_registry.to_csv(
        OUTPUT_DIR
        / "06_nested_split_registry.csv",
        index=False,
    )

    oof_predictions.to_parquet(
        OUTPUT_DIR
        / "07_all_variant_OOF_predictions.parquet",
        index=False,
    )

    primary_summary = (
        variant_summary.loc[
            variant_summary[
                "variant"
            ]
            == PRIMARY_VARIANT
        ]
        .copy()
    )

    primary_summary.to_csv(
        OUTPUT_DIR
        / "08_primary_variant_summary.csv",
        index=False,
    )

    expected_rows = int(
        train_raw[
            "GROUP_FOLD"
        ].isin(
            ACTIVE_OUTER_FOLDS
        ).sum()
    )

    stage_gate = {
        "run_mode": RUN_MODE,
        "official_test_unused": True,
        "consumed_official_test_path_never_read": True,
        "exact_legacy_reference_gate_passed": bool(
            legacy_reference_gate[
                "overall_reproduction_passed"
            ]
        ),
        "outer_split_group": "WELL",
        "inner_gate_split_group": "WELL",
        "expected_outer_folds": int(
            len(
                ACTIVE_OUTER_FOLDS
            )
        ),
        "observed_outer_folds": int(
            fold_metrics[
                "outer_fold"
            ].nunique()
        ),
        "outer_fold_count_correct": bool(
            fold_metrics[
                "outer_fold"
            ].nunique()
            == len(
                ACTIVE_OUTER_FOLDS
            )
        ),
        "expected_OOF_rows": (
            expected_rows
        ),
        "observed_OOF_rows": int(
            len(
                oof_predictions
            )
        ),
        "OOF_row_count_correct": bool(
            len(
                oof_predictions
            )
            == expected_rows
        ),
        "OOF_row_ids_unique": bool(
            not oof_predictions[
                "ORIGINAL_ROW_ID"
            ].duplicated().any()
        ),
        "all_fold_gates_passed": bool(
            all(
                row[
                    "fold_passed"
                ]
                for row in (
                    all_fold_gate_rows
                )
            )
        ),
        "legacy_full_RF_preserved": True,
        "TCN_raw_XY_excluded": True,
        "TCN_causal_engineered_features_preserved": True,
        "geo_support_context_included": True,
        "nested_meta_training_used": True,
        "inner_gate_labels_not_used_for_TCN_training_or_epoch_selection": True,
        "primary_variant_predeclared": (
            PRIMARY_VARIANT
        ),
        "primary_variant_present": bool(
            PRIMARY_VARIANT
            in set(
                variant_summary[
                    "variant"
                ]
            )
        ),
    }

    stage_gate[
        "overall_stage_passed"
    ] = bool(
        stage_gate[
            "official_test_unused"
        ]
        and stage_gate[
            "consumed_official_test_path_never_read"
        ]
        and stage_gate[
            "exact_legacy_reference_gate_passed"
        ]
        and stage_gate[
            "outer_fold_count_correct"
        ]
        and stage_gate[
            "OOF_row_count_correct"
        ]
        and stage_gate[
            "OOF_row_ids_unique"
        ]
        and stage_gate[
            "all_fold_gates_passed"
        ]
        and stage_gate[
            "legacy_full_RF_preserved"
        ]
        and stage_gate[
            "TCN_raw_XY_excluded"
        ]
        and stage_gate[
            "TCN_causal_engineered_features_preserved"
        ]
        and stage_gate[
            "geo_support_context_included"
        ]
        and stage_gate[
            "nested_meta_training_used"
        ]
        and stage_gate[
            "inner_gate_labels_not_used_for_TCN_training_or_epoch_selection"
        ]
        and stage_gate[
            "primary_variant_present"
        ]
    )

    with open(
        OUTPUT_DIR
        / "09_stage_gate.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            json_safe(
                stage_gate
            ),
            file,
            ensure_ascii=False,
            indent=2,
        )

    run_configuration = {
        "stage": (
            "08_legacy_TCN_geo_support_aware_stacking"
        ),
        "experiment_signature": (
            EXPERIMENT_SIGNATURE
        ),
        "run_mode": RUN_MODE,
        "official_test_used": False,
        "primary_variant": (
            PRIMARY_VARIANT
        ),
        "model_variants": (
            MODEL_VARIANTS
        ),
        "nested_split_registry": (
            split_registry.to_dict(
                orient="records"
            )
        ),
        "experiment_configuration": (
            EXPERIMENT_CONFIGURATION
        ),
        "TCN_training": {
            "gate_fold_used_for_training": False,
            "gate_fold_used_for_epoch_selection": False,
            "fixed_epochs": int(
                MAX_EPOCHS_INNER
            ),
            "inner_steps_per_epoch": int(
                STEPS_PER_EPOCH_INNER
            ),
            "outer_steps_per_epoch": int(
                STEPS_PER_EPOCH_OUTER
            ),
            "batch_size": int(
                BATCH_SIZE
            ),
        },
        "meta_model": {
            "type": (
                "multinomial_logistic_regression"
            ),
            "C": float(
                META_LOGISTIC_C
            ),
            "class_weight": "balanced",
            "well_balanced_sample_weights": True,
            "features": (
                meta_bundle.get(
                    "feature_names",
                    [],
                )
                if "meta_bundle" in globals()
                else []
            ),
        },
    }

    with open(
        OUTPUT_DIR
        / "10_run_configuration.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            json_safe(
                run_configuration
            ),
            file,
            ensure_ascii=False,
            indent=2,
        )

    environment = {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scipy": scipy.__version__,
        "sklearn": sklearn.__version__,
        "tensorflow": tf.__version__,
    }

    with open(
        OUTPUT_DIR
        / "99_environment.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            environment,
            file,
            ensure_ascii=False,
            indent=2,
        )

    print(
        json.dumps(
            stage_gate,
            indent=2,
        )
    )

    if not stage_gate[
        "overall_stage_passed"
    ]:
        raise RuntimeError(
            "Stage 08 gate failed."
        )

    print(
        "All outputs saved in:"
    )

    print(
        OUTPUT_DIR
    )

{
  "run_mode": "pilot",
  "official_test_unused": true,
  "consumed_official_test_path_never_read": true,
  "exact_legacy_reference_gate_passed": true,
  "outer_split_group": "WELL",
  "inner_gate_split_group": "WELL",
  "expected_outer_folds": 1,
  "observed_outer_folds": 1,
  "outer_fold_count_correct": true,
  "expected_OOF_rows": 232674,
  "observed_OOF_rows": 232674,
  "OOF_row_count_correct": true,
  "OOF_row_ids_unique": true,
  "all_fold_gates_passed": true,
  "legacy_full_RF_preserved": true,
  "TCN_raw_XY_excluded": true,
  "TCN_causal_engineered_features_preserved": true,
  "geo_support_context_included": true,
  "nested_meta_training_used": true,
  "inner_gate_labels_not_used_for_TCN_training_or_epoch_selection": true,
  "primary_variant_predeclared": "support_aware_stacking",
  "primary_variant_present": true,
  "overall_stage_passed": true
}
All outputs saved in:
<LOCAL_PATH_REDACTED>


# 17. Reused final-test benchmark helpers

In [26]:
def train_final_TCN_and_predict_reused_test(
    raw_full_train,
    raw_test,
    fixed_epoch_count,
):
    return train_outer_tcn_and_predict(
        raw_outer_train=raw_full_train,
        raw_outer_validation=raw_test,
        best_epoch=int(
            fixed_epoch_count
        ),
        outer_fold=999,
    )


def load_all_nested_meta_models():
    bundles = []

    for outer_fold in [
        1,
        2,
        3,
        4,
        5,
    ]:
        meta_path = fold_cache_paths(
            outer_fold
        )[
            "meta_model"
        ]

        if not meta_path.exists():
            raise FileNotFoundError(
                "Missing nested meta-model cache:\n"
                + str(
                    meta_path
                )
                + "\nRun RUN_MODE='full_cv' first."
            )

        bundle = joblib.load(
            meta_path
        )

        bundles.append({
            "outer_fold": int(
                outer_fold
            ),
            "bundle": bundle,
            "path": str(
                meta_path
            ),
        })

    return bundles


def ensemble_nested_meta_predictions(
    meta_bundles,
    meta_features,
):
    probability_parts = []

    for item in meta_bundles:
        probabilities = (
            predict_support_aware_meta_model(
                item[
                    "bundle"
                ],
                meta_features,
            )
        )

        probability_parts.append(
            probabilities
        )

    stacked = np.mean(
        np.stack(
            probability_parts,
            axis=0,
        ),
        axis=0,
    )

    return {
        "probabilities": (
            normalize_probability_rows(
                stacked
            )
        ),
        "member_probabilities": (
            probability_parts
        ),
    }


def calculate_variant_raw_and_refined_results(
    metadata,
    variant_probabilities,
):
    summary_rows = []
    per_class_frames = []
    per_well_frames = []
    prediction_frame = metadata.copy()

    y_true = metadata[
        "LITHOLOGY_ENCODED"
    ].to_numpy(
        dtype=int
    )

    for variant, probabilities in (
        variant_probabilities.items()
    ):
        probabilities = (
            normalize_probability_rows(
                probabilities
            )
        )

        (
            raw_predictions,
            refined_predictions,
            refinement_mask,
        ) = apply_legacy_chalk_refinement(
            probabilities
        )

        for output_type, predictions in [
            (
                "raw",
                raw_predictions,
            ),
            (
                "legacy_chalk_refined",
                refined_predictions,
            ),
        ]:
            result_name = (
                f"{variant}"
                f"__{output_type}"
            )

            metrics = calculate_metrics(
                y_true,
                predictions,
                probabilities=(
                    probabilities
                    if output_type
                    == "raw"
                    else None
                ),
            )

            summary_rows.append({
                "variant": variant,
                "output_type": (
                    output_type
                ),
                "result_name": (
                    result_name
                ),
                "is_predeclared_primary": bool(
                    variant
                    == PRIMARY_VARIANT
                    and output_type
                    == "raw"
                ),
                **metrics,
            })

            per_class = (
                calculate_per_class_metrics(
                    y_true,
                    predictions,
                )
            )

            per_class[
                "variant"
            ] = variant

            per_class[
                "output_type"
            ] = output_type

            per_class_frames.append(
                per_class
            )

            per_well = (
                calculate_generic_per_well_metrics(
                    metadata,
                    probabilities,
                    predictions,
                    variant=(
                        result_name
                    ),
                    outer_fold=0,
                )
            )

            per_well[
                "output_type"
            ] = output_type

            per_well_frames.append(
                per_well
            )

            prediction_frame[
                f"{result_name}"
                "__prediction_index"
            ] = predictions

        prediction_frame[
            f"{variant}"
            "__legacy_refinement_modified"
        ] = refinement_mask

        for class_index, class_code in enumerate(
            CLASS_CODES
        ):
            prediction_frame[
                f"{variant}"
                f"__prob_{class_code}"
            ] = probabilities[
                :,
                class_index,
            ].astype(
                np.float32
            )

    return {
        "summary": pd.DataFrame(
            summary_rows
        ),
        "per_class": pd.concat(
            per_class_frames,
            ignore_index=True,
        ),
        "per_well": pd.concat(
            per_well_frames,
            ignore_index=True,
        ),
        "predictions": (
            prediction_frame
        ),
    }

# 18. Reused final-test benchmark execution

In [27]:
if RUN_MODE == "final_test_reuse":
    with open(
        FULL_CV_STAGE_GATE_PATH,
        "r",
        encoding="utf-8",
    ) as file:
        full_cv_gate = json.load(
            file
        )

    if not full_cv_gate.get(
        "overall_stage_passed",
        False,
    ):
        raise RuntimeError(
            "Stage 08 full_cv gate did not pass."
        )

    if int(
        full_cv_gate.get(
            "observed_outer_folds",
            0,
        )
    ) != 5:
        raise RuntimeError(
            "A complete five-fold full_cv run is required."
        )

    full_cv_fold_metrics = pd.read_csv(
        FULL_CV_FOLD_METRICS_PATH
    )

    full_cv_variant_summary = pd.read_csv(
        FULL_CV_VARIANT_SUMMARY_PATH
    )

    fixed_epoch_values = sorted(
        pd.to_numeric(
            full_cv_fold_metrics[
                "fixed_tcn_epochs"
            ],
            errors="raise",
        )
        .dropna()
        .astype(
            int
        )
        .unique()
        .tolist()
    )

    if len(
        fixed_epoch_values
    ) != 1:
        raise RuntimeError(
            "Expected one predeclared TCN epoch count "
            "across full_cv, observed: "
            + str(
                fixed_epoch_values
            )
        )

    final_fixed_tcn_epochs = int(
        fixed_epoch_values[
            0
        ]
    )

    test_raw = load_csv_full(
        REUSED_FINAL_TEST_PATH
    )

    if TARGET_COLUMN not in (
        test_raw.columns
    ):
        raise RuntimeError(
            "The reused test file has no target labels."
        )

    test_raw[
        "ORIGINAL_ROW_ID"
    ] = np.arange(
        len(
            test_raw
        ),
        dtype=np.int64,
    )

    raw_full_train = (
        train_raw.drop(
            columns=[
                "GROUP_FOLD"
            ]
        )
        .copy()
        .reset_index(
            drop=True
        )
    )

    raw_test = (
        test_raw.copy()
        .reset_index(
            drop=True
        )
    )

    train_wells = set(
        raw_full_train[
            "WELL"
        ].dropna()
    )

    test_wells = set(
        raw_test[
            "WELL"
        ].dropna()
    )

    train_test_overlap = (
        train_wells
        & test_wells
    )

    if train_test_overlap:
        raise RuntimeError(
            "Train/test WELL overlap detected: "
            + str(
                sorted(
                    train_test_overlap
                )
            )
        )

    test_metadata = make_metadata_frame(
        raw_test
    )

    # --------------------------------------------------------
    # Final exact Legacy expert on all training wells.
    # --------------------------------------------------------

    final_legacy_result = (
        train_exact_legacy_full_and_predict(
            raw_full_train,
            raw_test,
        )
    )

    final_legacy_probabilities = (
        final_legacy_result[
            "probabilities"
        ]
    )

    if not np.array_equal(
        final_legacy_result[
            "metadata"
        ][
            "ORIGINAL_ROW_ID"
        ].to_numpy(
            dtype=np.int64
        ),
        test_metadata[
            "ORIGINAL_ROW_ID"
        ].to_numpy(
            dtype=np.int64
        ),
    ):
        raise RuntimeError(
            "Final Legacy test row order mismatch."
        )

    final_legacy_result[
        "resampling_audit"
    ].to_csv(
        OUTPUT_DIR
        / "01_final_train_resampling_audit.csv",
        index=False,
    )

    # --------------------------------------------------------
    # Final TCN expert on all training wells.
    # --------------------------------------------------------

    final_tcn_result = (
        train_final_TCN_and_predict_reused_test(
            raw_full_train,
            raw_test,
            fixed_epoch_count=(
                final_fixed_tcn_epochs
            ),
        )
    )

    final_tcn_probabilities = (
        final_tcn_result[
            "probabilities"
        ]
    )

    if not np.array_equal(
        final_tcn_result[
            "frame"
        ][
            "ORIGINAL_ROW_ID"
        ].to_numpy(
            dtype=np.int64
        ),
        test_metadata[
            "ORIGINAL_ROW_ID"
        ].to_numpy(
            dtype=np.int64
        ),
    ):
        raise RuntimeError(
            "Final TCN test row order mismatch."
        )

    # --------------------------------------------------------
    # Geo context from all training wells.
    # --------------------------------------------------------

    final_geo_bank = (
        build_geo_prior_bank(
            raw_full_train
        )
    )

    final_geo_result = compute_geo_prior(
        raw_test,
        final_geo_bank,
        k_neighbors=(
            GEO_K_NEIGHBORS
        ),
        vertical_half_window=(
            GEO_VERTICAL_HALF_WINDOW
        ),
    )

    final_meta_features = (
        build_meta_features(
            legacy_probabilities=(
                final_legacy_probabilities
            ),
            tcn_probabilities=(
                final_tcn_probabilities
            ),
            geo_result=(
                final_geo_result
            ),
            geo_bank=(
                final_geo_bank
            ),
        )
    )

    # The five nested meta-models were trained without the outer
    # held-out wells. Their probabilities are averaged.
    nested_meta_bundles = (
        load_all_nested_meta_models()
    )

    final_stacking_result = (
        ensemble_nested_meta_predictions(
            nested_meta_bundles,
            final_meta_features[
                "X"
            ],
        )
    )

    final_stacked_probabilities = (
        final_stacking_result[
            "probabilities"
        ]
    )

    final_simple_average = (
        normalize_probability_rows(
            0.5
            * final_legacy_probabilities
            + 0.5
            * final_tcn_probabilities
        )
    )

    final_variant_probabilities = {
        "legacy_only": (
            final_legacy_probabilities
        ),
        "tcn_only": (
            final_tcn_probabilities
        ),
        "simple_average": (
            final_simple_average
        ),
        "support_aware_stacking": (
            final_stacked_probabilities
        ),
    }

    final_results = (
        calculate_variant_raw_and_refined_results(
            test_metadata,
            final_variant_probabilities,
        )
    )

    final_summary = final_results[
        "summary"
    ]

    final_per_class = final_results[
        "per_class"
    ]

    final_per_well = final_results[
        "per_well"
    ]

    final_predictions = final_results[
        "predictions"
    ]

    final_predictions[
        "geo_reliability"
    ] = final_geo_result[
        "reliability_base"
    ]

    final_predictions[
        "geo_nearest_distance"
    ] = final_geo_result[
        "nearest_distance"
    ]

    final_predictions[
        "geo_support_wells"
    ] = final_geo_result[
        "support_wells"
    ]

    final_predictions[
        "geo_average_rows"
    ] = final_geo_result[
        "average_rows"
    ]

    final_predictions[
        "geo_agreement"
    ] = final_geo_result[
        "agreement"
    ]

    final_predictions[
        "geo_formation_specific"
    ] = final_geo_result[
        "formation_specific"
    ]

    final_summary.to_csv(
        OUTPUT_DIR
        / "02_reused_final_test_summary.csv",
        index=False,
    )

    final_per_class.to_csv(
        OUTPUT_DIR
        / "03_reused_final_test_per_class.csv",
        index=False,
    )

    final_per_well.to_csv(
        OUTPUT_DIR
        / "04_reused_final_test_per_well.csv",
        index=False,
    )

    final_predictions.to_parquet(
        OUTPUT_DIR
        / "05_reused_final_test_predictions.parquet",
        index=False,
    )

    full_cv_variant_summary.to_csv(
        OUTPUT_DIR
        / "06_full_cv_variant_summary_frozen.csv",
        index=False,
    )

    comparison_frames = []

    if ARTICLE_TEST_SUMMARY_PATH.exists():
        article_summary = pd.read_csv(
            ARTICLE_TEST_SUMMARY_PATH
        )

        article_summary[
            "comparison_source"
        ] = (
            "02A_article_exact_legacy"
        )

        comparison_frames.append(
            article_summary
        )

    model_test_summary = (
        final_summary.copy()
    )

    model_test_summary[
        "comparison_source"
    ] = (
        "08_reused_final_test"
    )

    comparison_frames.append(
        model_test_summary
    )

    comparison_table = pd.concat(
        comparison_frames,
        ignore_index=True,
        sort=False,
    )

    comparison_table.to_csv(
        OUTPUT_DIR
        / "07_comparison_with_article_test.csv",
        index=False,
    )

    display(
        comparison_table
    )

    probability_checks = {}

    for variant, probabilities in (
        final_variant_probabilities.items()
    ):
        probability_checks[
            variant
        ] = {
            "finite": bool(
                np.isfinite(
                    probabilities
                ).all()
            ),
            "sum_to_one": bool(
                np.allclose(
                    probabilities.sum(
                        axis=1
                    ),
                    1.0,
                    rtol=1e-6,
                    atol=1e-6,
                )
            ),
        }

    final_gate = {
        "run_mode": (
            RUN_MODE
        ),
        "test_role": (
            "reused_final_test_benchmark"
        ),
        "independent_untouched_evaluation": False,
        "full_cv_completed_before_test_run": True,
        "five_fold_full_cv_gate_passed": True,
        "primary_variant_predeclared": (
            PRIMARY_VARIANT
        ),
        "TCN_epoch_count_from_frozen_full_cv": int(
            final_fixed_tcn_epochs
        ),
        "five_nested_meta_models_loaded": bool(
            len(
                nested_meta_bundles
            )
            == 5
        ),
        "train_test_WELL_overlap_zero": bool(
            len(
                train_test_overlap
            )
            == 0
        ),
        "test_rows": int(
            len(
                test_metadata
            )
        ),
        "test_wells": int(
            test_metadata[
                "WELL"
            ].nunique()
        ),
        "legacy_full_RF_preserved": True,
        "TCN_raw_XY_excluded": True,
        "TCN_causal_features_preserved": True,
        "geo_context_training_labels_only": True,
        "no_test_labels_used_for_training": True,
        "no_test_labels_used_for_model_selection": True,
        "probability_checks": (
            probability_checks
        ),
    }

    final_gate[
        "overall_reused_test_stage_passed"
    ] = bool(
        final_gate[
            "full_cv_completed_before_test_run"
        ]
        and final_gate[
            "five_fold_full_cv_gate_passed"
        ]
        and final_gate[
            "five_nested_meta_models_loaded"
        ]
        and final_gate[
            "train_test_WELL_overlap_zero"
        ]
        and final_gate[
            "legacy_full_RF_preserved"
        ]
        and final_gate[
            "TCN_raw_XY_excluded"
        ]
        and final_gate[
            "TCN_causal_features_preserved"
        ]
        and final_gate[
            "geo_context_training_labels_only"
        ]
        and final_gate[
            "no_test_labels_used_for_training"
        ]
        and final_gate[
            "no_test_labels_used_for_model_selection"
        ]
        and all(
            check[
                "finite"
            ]
            and check[
                "sum_to_one"
            ]
            for check in (
                probability_checks.values()
            )
        )
    )

    with open(
        OUTPUT_DIR
        / "08_reused_final_test_gate.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            json_safe(
                final_gate
            ),
            file,
            ensure_ascii=False,
            indent=2,
        )

    final_run_configuration = {
        "stage": (
            "08_legacy_TCN_geo_support_aware_stacking"
        ),
        "experiment_signature": (
            EXPERIMENT_SIGNATURE
        ),
        "test_role": (
            "reused_final_test_benchmark"
        ),
        "primary_variant": (
            PRIMARY_VARIANT
        ),
        "fixed_tcn_epochs": int(
            final_fixed_tcn_epochs
        ),
        "meta_ensemble_outer_folds": [
            item[
                "outer_fold"
            ]
            for item in (
                nested_meta_bundles
            )
        ],
        "final_TCN_weights_path": (
            final_tcn_result[
                "weights_path"
            ]
        ),
        "geo_vertical_column": (
            final_geo_bank[
                "vertical_column"
            ]
        ),
        "test_was_previously_consumed": True,
        "do_not_tune_after_viewing_results": True,
    }

    with open(
        OUTPUT_DIR
        / "09_reused_final_test_run_configuration.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            json_safe(
                final_run_configuration
            ),
            file,
            ensure_ascii=False,
            indent=2,
        )

    sentinel = {
        "reused_final_test_consumed": True,
        "experiment_signature": (
            EXPERIMENT_SIGNATURE
        ),
        "primary_variant": (
            PRIMARY_VARIANT
        ),
        "instruction": (
            "Do not tune Stage 08 using this reused test result."
        ),
    }

    with open(
        REUSED_TEST_SENTINEL_PATH,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            json_safe(
                sentinel
            ),
            file,
            ensure_ascii=False,
            indent=2,
        )

    print(
        json.dumps(
            final_gate,
            indent=2,
        )
    )

    if not final_gate[
        "overall_reused_test_stage_passed"
    ]:
        raise RuntimeError(
            "Reused final-test stage gate failed."
        )

    print(
        "Reused final-test outputs saved in:"
    )

    print(
        OUTPUT_DIR
    )